# **Model Evaluation for Constants**

In order to evaluate the initial LLM model's performance on the task of constant redefinition, we leverage another LLM model as an evaluator. We calculate the percentages of **Correct Responses**, **Anchored Responses**, and **Completely Wrong Responses**. For the Completely Wrong Responses, we further analyze whether the model's error is due to an incorrect result, a blank answer, or a refusal to answer. 

In [ ]:
!pip install --upgrade boto3 
!pip install --upgrade botocore
!pip install langchain

For the evaluator, we use the model anthropic.claude-3-5-sonnet-20241022-v2:0, which we access via AWS Bedrock. To establish access, you must provide your AWS access key credentials.

In [ ]:
# Import required libraries
import boto3
import json
from botocore.exceptions import ClientError
from langchain.prompts import PromptTemplate

# AWS Bedrock setup 
client = boto3.client(
    service_name="bedrock-runtime",
    aws_access_key_id='your_aws_access_key_id',
    aws_secret_access_key='your_aws_secret_access_key',
    region_name="us-west-2"
)

In [ ]:
claude_model_id = "anthropic.claude-3-5-sonnet-20241022-v2:0"

def get_claude_response(messages):
    
    native_request = {
        "messages": messages,
        "max_tokens": 512,
        "temperature": 0.5,
        "anthropic_version": "bedrock-2023-05-31"
    }

    request = json.dumps(native_request)

    try:
        response = client.invoke_model(modelId=claude_model_id, body=request)
        
        model_response = json.loads(response["body"].read())

        return model_response["content"][0]["text"]
    
    except (ClientError, Exception) as e:
        print(f"ERROR: Can't invoke 'anthropic.claude-3'. Reason: {e}")
        return None

In [ ]:
def get_claude_answer(prompt_template, llm_answer, real_answer):
    claude_messages = [
        {"role": "user", "content": comparison_prompt.format(llm_answer=llm_answer, real_answer=real_answer)},
    ]
    claude_ans = get_claude_response(claude_messages)
    return claude_ans

# **Datasets**

We extract the correct answers from each of the six datasets that we used for redefining constants (three in free-form (FF) format and three in multiple-choice (MC) format, each corresponding to a different difficulty level). To evaluate the model, we also extract the responses from the .csv files that we obtained from constant_redefinition.ipynb.

In [2]:
import os
import pandas as pd

is_kaggle = os.path.exists('/kaggle/input')

# If you're running this notebook in the Kaggle environment you need to manually upload the files contained in the 'input' folder
#   Upload these files to the Kaggle "Input" section before running the code.
#   Make sure the folder structure matches the expected directories:
#   - 'constants-level1' folder containing 'constants - level1.csv' and 'constants - level1_mc.csv'
#   - 'constants-level2' folder containing 'constants - level2.csv' and 'constants - level2_mc.csv'
#   - 'constants-level3' folder containing 'constants - level3.csv' and 'constants - level3_mc.csv'

if is_kaggle:
    input_dir = '/kaggle/input'
else:
    input_dir = os.path.join(os.getcwd(), 'input')

level1_dir = os.path.join(input_dir, "constants-level1")
level2_dir = os.path.join(input_dir, "constants-level2")
level3_dir = os.path.join(input_dir, "constants-level3")

file_level1 = ["constants - level1.csv", "constants - level1_mc.csv"]
file_level2 = ["constants - level2.csv", "constants - level2_mc.csv"]
file_level3 = ["constants - level3.csv", "constants - level3_mc.csv"]


df1 = pd.read_csv(os.path.join(level1_dir, file_level1[0]))  
df1_mc = pd.read_csv(os.path.join(level1_dir, file_level1[1]))  

df2 = pd.read_csv(os.path.join(level2_dir, file_level2[0]))  
df2_mc = pd.read_csv(os.path.join(level2_dir, file_level2[1])) 

df3 = pd.read_csv(os.path.join(level3_dir, file_level3[0]))  
df3_mc = pd.read_csv(os.path.join(level3_dir, file_level3[1])) 

In [4]:
correct_answers1 = [
    {
        "X": row["X"],
        "real_value": row["real_value"],
        "question": row["question"],
        "answer_real": row["answer_real"],
        "Y1": row["Y1"],
        "answer1": row["answer1"],
        "Y2": row["Y2"],
        "answer2": row["answer2"],
        "Y3": row["Y3"],
        "answer3": row["answer3"],
        "C1": row["C1"],
        "answerC1": row["answerC1"],
        "C2": row["C2"],
        "answerC2": row["answerC2"]
    }
    for _, row in df1.iterrows()
]
correct_answers1[:2]

[{'X': 'pi',
  'real_value': '3.14159',
  'question': 'What is the first digit of pi?',
  'answer_real': '3',
  'Y1': '4.5',
  'answer1': '4',
  'Y2': '500',
  'answer2': '5',
  'Y3': '-10',
  'answer3': '1',
  'C1': 'phi',
  'answerC1': '1',
  'C2': "the Planck's constant",
  'answerC2': '6'},
 {'X': 'e',
  'real_value': '2.71828',
  'question': 'What is the first digit of e?',
  'answer_real': '2',
  'Y1': '9',
  'answer1': '9',
  'Y2': '1300',
  'answer2': '1',
  'Y3': '1.5×10^(−12)',
  'answer3': '1',
  'C1': 'pi',
  'answerC1': '3',
  'C2': 'the Boltzmann constant',
  'answerC2': '1'}]

In [5]:
correct_answers2 = [
    {
        "X": row["X"],
        "real_value": row["real_value"],
        "question": row["question"],
        "answer_real": row["answer_real"],
        "Y1": row["Y1"],
        "answer1": row["answer1"],
        "Y2": row["Y2"],
        "answer2": row["answer2"],
        "Y3": row["Y3"],
        "answer3": row["answer3"],
        "C1": row["C1"],
        "answerC1": row["answerC1"],
        "C2": row["C2"],
        "answerC2": row["answerC2"]
    }
    for _, row in df2.iterrows()
]
correct_answers2[:2]

[{'X': 'pi',
  'real_value': '3.14159',
  'question': 'What is pi multiplied by 3?',
  'answer_real': '9.4248',
  'Y1': '4.5',
  'answer1': '13.5',
  'Y2': '500',
  'answer2': '1500',
  'Y3': '-10',
  'answer3': '-30',
  'C1': 'phi',
  'answerC1': '4.854',
  'C2': "the Planck's constant",
  'answerC2': '1.9878×10^(−33)'},
 {'X': 'e',
  'real_value': '2.71828',
  'question': 'What is e^2?',
  'answer_real': '7.389',
  'Y1': '9',
  'answer1': '81',
  'Y2': '1300',
  'answer2': '1,690,000',
  'Y3': '1.5×10^(−12)',
  'answer3': '2.25×10^(−24)',
  'C1': 'pi',
  'answerC1': '9.8696',
  'C2': 'the Boltzmann constant',
  'answerC2': '1.904×10^(−46)'}]

In [7]:
correct_answers3 = [
    {
        "X": row["X"],
        "real_value": row["real_value"],
        "question": row["question"],
        "answer_real": row["answer_real"],
        "Y1": row["Y1"],
        "answer1": row["answer1"],
        "Y2": row["Y2"],
        "answer2": row["answer2"],
        "Y3": row["Y3"],
        "answer3": row["answer3"],
        "C1": row["C1"],
        "answerC1": row["answerC1"],
        "C2": row["C2"],
        "answerC2": row["answerC2"]
    }
    for _, row in df3.iterrows()
]
correct_answers3[:2]

[{'X': 'pi',
  'real_value': '3.14159',
  'question': "What is the Earth's surface area?",
  'answer_real': '510,064,472 square kilometers',
  'Y1': '4.5',
  'answer1': '730,397,538 square kilometers',
  'Y2': '500',
  'answer2': '81,155,282,000 square kilometers',
  'Y3': '-10',
  'answer3': '−1,623,105,640 square kilometers',
  'C1': 'phi',
  'answerC1': '262,642,778 square kilometers',
  'C2': "the Planck's constant",
  'answerC2': '1.08×10^(−25) square kilometers'},
 {'X': 'e',
  'real_value': '2.71828',
  'question': 'If a population grows continuously at a rate of 5% per year, by what factor will it increase in 10 years?',
  'answer_real': '1.649',
  'Y1': '9',
  'answer1': '3',
  'Y2': '1300',
  'answer2': '36.06',
  'Y3': '1.5×10^(−12)',
  'answer3': '1.22×10^(−6)',
  'C1': 'pi',
  'answerC1': '1.77245',
  'C2': 'the Boltzmann constant',
  'answerC2': '3.714×10^(−12) '}]

In [9]:
correct_answers1_mc = [
    {
        "X": row["X"],
        "real_value": row["real_value"],
        "question": row["question"],
        "answer_real": row["answer_real"],
        "Y1": row["Y1"],
        "answer1": row["answer1"],
        "answer1_real": row["answer1_real"],
        "Y2": row["Y2"],
        "answer2": row["answer2"],
        "answer2_real": row["answer2_real"],
        "Y3": row["Y3"],
        "answer3": row["answer3"],
        "answer3_real": row["answer3_real"],
        "C1": row["C1"],
        "answerC1": row["answerC1"],
        "answerC1_real": row["answerC1_real"],
        "C2": row["C2"],
        "answerC2": row["answerC2"],
        "answerC2_real": row["answerC2_real"]
    }
    for _, row in df1_mc.iterrows()
]
correct_answers1_mc[:2]

[{'X': 'pi',
  'real_value': '3.14159',
  'question': 'What is the first digit of pi?',
  'answer_real': '3',
  'Y1': '4.5',
  'answer1': 'C',
  'answer1_real': 'B',
  'Y2': '500',
  'answer2': 'D',
  'answer2_real': 'B',
  'Y3': '-10',
  'answer3': 'A',
  'answer3_real': 'C',
  'C1': '4',
  'answerC1': 'A',
  'answerC1_real': 'C',
  'C2': '4',
  'answerC2': 'D',
  'answerC2_real': 'A'},
 {'X': 'e',
  'real_value': '2.71828',
  'question': 'What is the first digit of e?',
  'answer_real': '2',
  'Y1': '9',
  'answer1': 'D',
  'answer1_real': 'A',
  'Y2': '1300',
  'answer2': 'A',
  'answer2_real': 'B',
  'Y3': '1.5×10^(−12)',
  'answer3': 'A',
  'answer3_real': 'B',
  'C1': '7',
  'answerC1': 'C',
  'answerC1_real': 'B',
  'C2': '3',
  'answerC2': 'A',
  'answerC2_real': 'B'}]

In [10]:
correct_answers2_mc = [
    {
        "X": row["X"],
        "real_value": row["real_value"],
        "question": row["question"],
        "answer_real": row["answer_real"],
        "Y1": row["Y1"],
        "answer1": row["answer1"],
        "answer1_real": row["answer1_real"],
        "Y2": row["Y2"],
        "answer2": row["answer2"],
        "answer2_real": row["answer2_real"],
        "Y3": row["Y3"],
        "answer3": row["answer3"],
        "answer3_real": row["answer3_real"],
        "C1": row["C1"],
        "answerC1": row["answerC1"],
        "answerC1_real": row["answerC1_real"],
        "C2": row["C2"],
        "answerC2": row["answerC2"],
        "answerC2_real": row["answerC2_real"]
    }
    for _, row in df2_mc.iterrows()
]
correct_answers2_mc[:2]

[{'X': 'pi',
  'real_value': '3.14159',
  'question': 'What is pi multiplied by 3?',
  'answer_real': '9.4248',
  'Y1': '4.5',
  'answer1': 'C',
  'answer1_real': 'B',
  'Y2': '500',
  'answer2': 'D',
  'answer2_real': 'B',
  'Y3': '-10',
  'answer3': 'C',
  'answer3_real': 'B',
  'C1': '13.5',
  'answerC1': 'B',
  'answerC1_real': 'D',
  'C2': '500',
  'answerC2': 'B',
  'answerC2_real': 'D'},
 {'X': 'e',
  'real_value': '2.71828',
  'question': 'What is e^2?',
  'answer_real': '7.389',
  'Y1': '9',
  'answer1': 'D',
  'answer1_real': 'A',
  'Y2': '1300',
  'answer2': 'D',
  'answer2_real': 'A',
  'Y3': '1.5×10^(−12)',
  'answer3': 'D',
  'answer3_real': 'A',
  'C1': '18',
  'answerC1': 'C',
  'answerC1_real': 'B',
  'C2': '1300',
  'answerC2': 'A',
  'answerC2_real': 'D'}]

In [11]:
correct_answers3_mc = [
    {
        "X": row["X"],
        "real_value": row["real_value"],
        "question": row["question"],
        "answer_real": row["answer_real"],
        "Y1": row["Y1"],
        "answer1": row["answer1"],
        "answer1_real": row["answer1_real"],
        "Y2": row["Y2"],
        "answer2": row["answer2"],
        "answer2_real": row["answer2_real"],
        "Y3": row["Y3"],
        "answer3": row["answer3"],
        "answer3_real": row["answer3_real"],
        "C1": row["C1"],
        "answerC1": row["answerC1"],
        "answerC1_real": row["answerC1_real"],
        "C2": row["C2"],
        "answerC2": row["answerC2"],
        "answerC2_real": row["answerC2_real"]
    }
    for _, row in df3_mc.iterrows()
]
correct_answers3_mc[:2]

[{'X': 'pi',
  'real_value': '3.14159',
  'question': "What is the Earth's surface area?",
  'answer_real': '510,064,472 square kilometers',
  'Y1': '4.5',
  'answer1': 'B',
  'answer1_real': 'A',
  'Y2': '500',
  'answer2': 'D',
  'answer2_real': 'B',
  'Y3': '-10',
  'answer3': 'C',
  'answer3_real': 'B',
  'C1': '730,397,538 square meters',
  'answerC1': 'B',
  'answerC1_real': 'D',
  'C2': '81,155,282 square kilometers',
  'answerC2': 'A',
  'answerC2_real': 'B'},
 {'X': 'e',
  'real_value': '2.71828',
  'question': 'If a population grows continuously at a rate of 5% per year, by what factor will it increase in 10 years?',
  'answer_real': '1.649',
  'Y1': '9',
  'answer1': 'D',
  'answer1_real': 'A',
  'Y2': '1300',
  'answer2': 'D',
  'answer2_real': 'A',
  'Y3': '1.5×10^(−12)',
  'answer3': 'D',
  'answer3_real': 'A',
  'C1': '81',
  'answerC1': 'C',
  'answerC1_real': 'B',
  'C2': '1300',
  'answerC2': 'D',
  'answerC2_real': 'A'}]

In [ ]:
# Here, you should provide the paths to the .csv files exported from the `constant_redefinition.ipynb` notebook. 
# These files contain the model's responses, and you need to replace the path in `read_csv` with the correct location of the exported .csv file.

df1_llm = pd.read_csv('/kaggle/input/titan-text-lite/titan_text_lite_results1.csv')
df1_llm.head()

In [ ]:
df2_llm = pd.read_csv('/kaggle/input/titan-text-lite/titan_text_lite_results2.csv')
df2_llm.head()

In [ ]:
df3_llm = pd.read_csv('/kaggle/input/titan-text-lite/titan_text_lite_results3.csv')
df3_llm.head()

In [ ]:
llm_answers1 = [
    {
        "no_redefinition": row["no_redefinition"],
        "Y1_redefinition_zero": row["Y1_redefinition_zero"],
        "Y1_redefinition_cot": row["Y1_redefinition_cot"],
        "Y1_redefinition_few": row["Y1_redefinition_few"],
        "Y2_redefinition_zero": row["Y2_redefinition_zero"],
        "Y2_redefinition_cot": row["Y2_redefinition_cot"],
        "Y2_redefinition_few": row["Y2_redefinition_few"],
        "Y3_redefinition_zero": row["Y3_redefinition_zero"],
        "Y3_redefinition_cot": row["Y3_redefinition_cot"],
        "Y3_redefinition_few": row["Y3_redefinition_few"],
        "C1_redefinition_zero": row["C1_redefinition_zero"],
        "C1_redefinition_cot": row["C1_redefinition_cot"],
        "C1_redefinition_few": row["C1_redefinition_few"],
        "C2_redefinition_zero": row["C2_redefinition_zero"],
        "C2_redefinition_cot": row["C2_redefinition_cot"],
        "C2_redefinition_few": row["C2_redefinition_few"],
        
        "Y1_redefinition_mc_zero": row["Y1_redefinition_mc_zero"],
        "Y1_redefinition_mc_cot": row["Y1_redefinition_mc_cot"],
        "Y1_redefinition_mc_few": row["Y1_redefinition_mc_few"],
        "Y2_redefinition_mc_zero": row["Y2_redefinition_mc_zero"],
        "Y2_redefinition_mc_cot": row["Y2_redefinition_mc_cot"],
        "Y2_redefinition_mc_few": row["Y2_redefinition_mc_few"],
        "Y3_redefinition_mc_zero": row["Y3_redefinition_mc_zero"],
        "Y3_redefinition_mc_cot": row["Y3_redefinition_mc_cot"],
        "Y3_redefinition_mc_few": row["Y3_redefinition_mc_few"],
        "C1_redefinition_mc_zero": row["C1_redefinition_mc_zero"],
        "C1_redefinition_mc_cot": row["C1_redefinition_mc_cot"],
        "C1_redefinition_mc_few": row["C1_redefinition_mc_few"],
        "C2_redefinition_mc_zero": row["C2_redefinition_mc_zero"],
        "C2_redefinition_mc_cot": row["C2_redefinition_mc_cot"],
        "C2_redefinition_mc_few": row["C2_redefinition_mc_few"],
    }
    for _, row in df1_llm.iterrows()
]
#llm_answers1[:2]

In [ ]:
llm_answers2 = [
    {
        "no_redefinition": row["no_redefinition"],
        "Y1_redefinition_zero": row["Y1_redefinition_zero"],
        "Y1_redefinition_cot": row["Y1_redefinition_cot"],
        "Y1_redefinition_few": row["Y1_redefinition_few"],
        "Y2_redefinition_zero": row["Y2_redefinition_zero"],
        "Y2_redefinition_cot": row["Y2_redefinition_cot"],
        "Y2_redefinition_few": row["Y2_redefinition_few"],
        "Y3_redefinition_zero": row["Y3_redefinition_zero"],
        "Y3_redefinition_cot": row["Y3_redefinition_cot"],
        "Y3_redefinition_few": row["Y3_redefinition_few"],
        "C1_redefinition_zero": row["C1_redefinition_zero"],
        "C1_redefinition_cot": row["C1_redefinition_cot"],
        "C1_redefinition_few": row["C1_redefinition_few"],
        "C2_redefinition_zero": row["C2_redefinition_zero"],
        "C2_redefinition_cot": row["C2_redefinition_cot"],
        "C2_redefinition_few": row["C2_redefinition_few"],
        
        "Y1_redefinition_mc_zero": row["Y1_redefinition_mc_zero"],
        "Y1_redefinition_mc_cot": row["Y1_redefinition_mc_cot"],
        "Y1_redefinition_mc_few": row["Y1_redefinition_mc_few"],
        "Y2_redefinition_mc_zero": row["Y2_redefinition_mc_zero"],
        "Y2_redefinition_mc_cot": row["Y2_redefinition_mc_cot"],
        "Y2_redefinition_mc_few": row["Y2_redefinition_mc_few"],
        "Y3_redefinition_mc_zero": row["Y3_redefinition_mc_zero"],
        "Y3_redefinition_mc_cot": row["Y3_redefinition_mc_cot"],
        "Y3_redefinition_mc_few": row["Y3_redefinition_mc_few"],
        "C1_redefinition_mc_zero": row["C1_redefinition_mc_zero"],
        "C1_redefinition_mc_cot": row["C1_redefinition_mc_cot"],
        "C1_redefinition_mc_few": row["C1_redefinition_mc_few"],
        "C2_redefinition_mc_zero": row["C2_redefinition_mc_zero"],
        "C2_redefinition_mc_cot": row["C2_redefinition_mc_cot"],
        "C2_redefinition_mc_few": row["C2_redefinition_mc_few"],
    }
    for _, row in df2_llm.iterrows()
]
#llm_answers2[:2]

In [ ]:
llm_answers3 = [
    {
        "no_redefinition": row["no_redefinition"],
        "Y1_redefinition_zero": row["Y1_redefinition_zero"],
        "Y1_redefinition_cot": row["Y1_redefinition_cot"],
        "Y1_redefinition_few": row["Y1_redefinition_few"],
        "Y2_redefinition_zero": row["Y2_redefinition_zero"],
        "Y2_redefinition_cot": row["Y2_redefinition_cot"],
        "Y2_redefinition_few": row["Y2_redefinition_few"],
        "Y3_redefinition_zero": row["Y3_redefinition_zero"],
        "Y3_redefinition_cot": row["Y3_redefinition_cot"],
        "Y3_redefinition_few": row["Y3_redefinition_few"],
        "C1_redefinition_zero": row["C1_redefinition_zero"],
        "C1_redefinition_cot": row["C1_redefinition_cot"],
        "C1_redefinition_few": row["C1_redefinition_few"],
        "C2_redefinition_zero": row["C2_redefinition_zero"],
        "C2_redefinition_cot": row["C2_redefinition_cot"],
        "C2_redefinition_few": row["C2_redefinition_few"],
        
        "Y1_redefinition_mc_zero": row["Y1_redefinition_mc_zero"],
        "Y1_redefinition_mc_cot": row["Y1_redefinition_mc_cot"],
        "Y1_redefinition_mc_few": row["Y1_redefinition_mc_few"],
        "Y2_redefinition_mc_zero": row["Y2_redefinition_mc_zero"],
        "Y2_redefinition_mc_cot": row["Y2_redefinition_mc_cot"],
        "Y2_redefinition_mc_few": row["Y2_redefinition_mc_few"],
        "Y3_redefinition_mc_zero": row["Y3_redefinition_mc_zero"],
        "Y3_redefinition_mc_cot": row["Y3_redefinition_mc_cot"],
        "Y3_redefinition_mc_few": row["Y3_redefinition_mc_few"],
        "C1_redefinition_mc_zero": row["C1_redefinition_mc_zero"],
        "C1_redefinition_mc_cot": row["C1_redefinition_mc_cot"],
        "C1_redefinition_mc_few": row["C1_redefinition_mc_few"],
        "C2_redefinition_mc_zero": row["C2_redefinition_mc_zero"],
        "C2_redefinition_mc_cot": row["C2_redefinition_mc_cot"],
        "C2_redefinition_mc_few": row["C2_redefinition_mc_few"],
    }
    for _, row in df3_llm.iterrows()
]
#llm_answers3[:2]

# **No Redefinition** 

For the No Redefinition part, we compare the model's response to the correct answer and calculate the resulting accuracy for each level of difficulty.

In [ ]:
from langchain.prompts import PromptTemplate

comparison_prompt = PromptTemplate(
    input_variables=["llm_answer", "real_answer"],
    template="""
You are tasked with comparing two answers: one provided by an LLM (the "LLM answer") and the correct answer (the "real answer"). Your job is to determine if the LLM answer matches the real answer.

The comparison should strictly focus on whether the LLM final answer conveys the same meaning or provide the same information as the correct answer. Minor differences in phrasing, wording, or structure are acceptable as long as the core meaning remains identical. For numerical results, differences due to rounding are acceptable as long as the values are reasonably close and within an acceptable margin of error.

**Instructions:**
1. Compare the LLM answer to the real answer carefully.
2. If the LLM answer matches the real answer, output:
   correct
3. If the LLM answer does not match the real answer, output:
   incorrect

Do not output explanations, justifications, or any additional text — only the word "correct" or "incorrect".

**LLM Answer:**  
{llm_answer}

**Real Answer:**  
{real_answer}

**Output:**
"""
)

In [ ]:
NR_res1 = []

for i in range(len(llm_answers1)):
    llm_answer = llm_answers1[i]["no_redefinition"]
    real_answer = correct_answers1[i]["answer_real"]
    NR_res1.append(get_claude_answer(comparison_prompt, llm_answer, real_answer))

In [ ]:
NR_res2 = []

for i in range(len(llm_answers2)):
    llm_answer = llm_answers2[i]["no_redefinition"]
    real_answer = correct_answers2[i]["answer_real"]
    NR_res2.append(get_claude_answer(comparison_prompt, llm_answer, real_answer))

In [ ]:
NR_res3 = []

for i in range(len(llm_answers3)):
    llm_answer = llm_answers3[i]["no_redefinition"]
    real_answer = correct_answers3[i]["answer_real"]
    NR_res3.append(get_claude_answer(comparison_prompt, llm_answer, real_answer))

In [ ]:
acc1=0
for item in NR_res1:
    if item=="correct":
        acc1 = acc1+1
acc1 = round(100*acc1/len(NR_res1), 2)
print(f"No Redefinition 1: {acc1}")

In [ ]:
acc2=0
for item in NR_res2:
    if item=="correct":
        acc2 = acc2+1
acc2 = round(100*acc2/len(NR_res2), 2)
print(f"No Redefinition 2: {acc2}")

In [ ]:
acc3=0
for item in NR_res3:
    if item=="correct":
        acc3 = acc3+1
acc3 = round(100*acc3/len(NR_res3), 2)
print(f"No Redefinition 3: {acc3}")

# **Redefinition**

For the Redefinition part, we compare the model's response to the correct answer both before and after the redefinition, in order to distinguish between Correct, Anchored, and Completely Wrong. 

In [ ]:
comparison_prompt_redefinition = PromptTemplate(
    input_variables=["llm_answer", "reference_answer_1", "reference_answer_2"],
    template="""
You are tasked with comparing an answer provided by an LLM (the "LLM answer") to two reference answers: "Reference Answer 1" and "Reference Answer 2". Your job is to determine if the LLM answer matches either of the two reference answers.

The comparison should strictly focus on whether the LLM final answer conveys the same meaning or provides the same information as one of the reference answers. Minor differences in phrasing, wording, or structure are acceptable as long as the core meaning remains identical. For numerical results, differences due to rounding are acceptable as long as the values are reasonably close and within an acceptable margin of error.

**Instructions:**
1. Compare the LLM answer carefully with "Reference Answer 1" and "Reference Answer 2".
2. If the LLM answer matches "Reference Answer 1", output:  
   first
3. If the LLM answer matches "Reference Answer 2", output:  
   second
4. If the LLM answer matches neither of the two, output:  
   none

Do not output explanations, justifications, or any additional text — only the words "first", "second", or "none".

**LLM Answer:**  
{llm_answer}

**Reference Answer 1:**  
{reference_answer_1}

**Reference Answer 2:**  
{reference_answer_2}

**Output:**
"""
)

In [ ]:
def get_claude_answer_redefinition(prompt_template, llm_answer, reference_answer_1, reference_answer_2):
    claude_messages = [
        {"role": "user", "content": comparison_prompt_redefinition.format(llm_answer=llm_answer, reference_answer_1=reference_answer_1, reference_answer_2=reference_answer_2)},
    ]
    claude_ans = get_claude_response(claude_messages)
    return claude_ans

In [ ]:
def compute_res(llm_answers, correct_answers, key1, key2):
    
    res=[]
    for i in range(len(llm_answers)):
        llm_answer = llm_answers[i][key1]
        reference_answer_1 = correct_answers[i][key2]
        reference_answer_2 = correct_answers[i]["answer_real"]
        res.append(get_claude_answer_redefinition(comparison_prompt_redefinition, llm_answer, reference_answer_1, reference_answer_2
        ))
        
    return res

In [ ]:
Y1_zero1=compute_res(llm_answers1, correct_answers1, "Y1_redefinition_zero", "answer1")
Y1_cot1=compute_res(llm_answers1, correct_answers1, "Y1_redefinition_cot", "answer1")
Y1_few1=compute_res(llm_answers1, correct_answers1, "Y1_redefinition_few", "answer1")

Y2_zero1=compute_res(llm_answers1, correct_answers1, "Y2_redefinition_zero", "answer2")
Y2_cot1=compute_res(llm_answers1, correct_answers1, "Y2_redefinition_cot", "answer2")
Y2_few1=compute_res(llm_answers1, correct_answers1, "Y2_redefinition_few", "answer2")

Y3_zero1=compute_res(llm_answers1, correct_answers1, "Y3_redefinition_zero", "answer3")
Y3_cot1=compute_res(llm_answers1, correct_answers1, "Y3_redefinition_cot", "answer3")
Y3_few1=compute_res(llm_answers1, correct_answers1, "Y3_redefinition_few", "answer3")

C1_zero1=compute_res(llm_answers1, correct_answers1, "C1_redefinition_zero", "answerC1")
C1_cot1=compute_res(llm_answers1, correct_answers1, "C1_redefinition_cot", "answerC1")
C1_few1=compute_res(llm_answers1, correct_answers1, "C1_redefinition_few", "answerC1")

C2_zero1=compute_res(llm_answers1, correct_answers1, "C2_redefinition_zero", "answerC2")
C2_cot1=compute_res(llm_answers1, correct_answers1, "C2_redefinition_cot", "answerC2")
C2_few1=compute_res(llm_answers1, correct_answers1, "C2_redefinition_few", "answerC2")

In [ ]:
Y1_zero2=compute_res(llm_answers2, correct_answers2, "Y1_redefinition_zero", "answer1")
Y1_cot2=compute_res(llm_answers2, correct_answers2, "Y1_redefinition_cot", "answer1")
Y1_few2=compute_res(llm_answers2, correct_answers2, "Y1_redefinition_few", "answer1")

Y2_zero2=compute_res(llm_answers2, correct_answers2, "Y2_redefinition_zero", "answer2")
Y2_cot2=compute_res(llm_answers2, correct_answers2, "Y2_redefinition_cot", "answer2")
Y2_few2=compute_res(llm_answers2, correct_answers2, "Y2_redefinition_few", "answer2")

Y3_zero2=compute_res(llm_answers2, correct_answers2, "Y3_redefinition_zero", "answer3")
Y3_cot2=compute_res(llm_answers2, correct_answers2, "Y3_redefinition_cot", "answer3")
Y3_few2=compute_res(llm_answers2, correct_answers2, "Y3_redefinition_few", "answer3")

C1_zero2=compute_res(llm_answers2, correct_answers2, "C1_redefinition_zero", "answerC1")
C1_cot2=compute_res(llm_answers2, correct_answers2, "C1_redefinition_cot", "answerC1")
C1_few2=compute_res(llm_answers2, correct_answers2, "C1_redefinition_few", "answerC1")

C2_zero2=compute_res(llm_answers2, correct_answers2, "C2_redefinition_zero", "answerC2")
C2_cot2=compute_res(llm_answers2, correct_answers2, "C2_redefinition_cot", "answerC2")
C2_few2=compute_res(llm_answers2, correct_answers2, "C2_redefinition_few", "answerC2")

In [ ]:
Y1_zero3=compute_res(llm_answers3, correct_answers3, "Y1_redefinition_zero", "answer1")
Y1_cot3=compute_res(llm_answers3, correct_answers3, "Y1_redefinition_cot", "answer1")
Y1_few3=compute_res(llm_answers3, correct_answers3, "Y1_redefinition_few", "answer1")

Y2_zero3=compute_res(llm_answers3, correct_answers3, "Y2_redefinition_zero", "answer2")
Y2_cot3=compute_res(llm_answers3, correct_answers3, "Y2_redefinition_cot", "answer2")
Y2_few3=compute_res(llm_answers3, correct_answers3, "Y2_redefinition_few", "answer2")

Y3_zero3=compute_res(llm_answers3, correct_answers3, "Y3_redefinition_zero", "answer3")
Y3_cot3=compute_res(llm_answers3, correct_answers3, "Y3_redefinition_cot", "answer3")
Y3_few3=compute_res(llm_answers3, correct_answers3, "Y3_redefinition_few", "answer3")

C1_zero3=compute_res(llm_answers3, correct_answers3, "C1_redefinition_zero", "answerC1")
C1_cot3=compute_res(llm_answers3, correct_answers3, "C1_redefinition_cot", "answerC1")
C1_few3=compute_res(llm_answers3, correct_answers3, "C1_redefinition_few", "answerC1")

C2_zero3=compute_res(llm_answers3, correct_answers3, "C2_redefinition_zero", "answerC2")
C2_cot3=compute_res(llm_answers3, correct_answers3, "C2_redefinition_cot", "answerC2")
C2_few3=compute_res(llm_answers3, correct_answers3, "C2_redefinition_few", "answerC2")

We further analyze the Completely Wrong Responses to inventigate the underlying causes for errors. We distinguish between **Wrong Result**, **Refusal to Answer**, and **Blank Answer**.

In [ ]:
analysis_prompt = PromptTemplate(
    input_variables=["llm_answer", "reference_answer_1", "reference_answer_2"],
    template="""
You are tasked with analyzing an LLM answer that does not match either of two reference answers: "Reference Answer 1" and "Reference Answer 2". Your job is to classify the LLM answer into one of the following categories:

1. **Wrong Answer**: The LLM provided an incorrect response to the question, either factually or logically.
2. **Blank Answer**: The LLM provided no substantive response, leaving the answer blank or completely empty.
3. **Refusal to Answer**: The LLM explicitly refused to answer the question, citing reasons such as the question being nonsensical, impossible to answer, or against its guidelines.

**Instructions:**
1. Analyze the LLM answer and determine which of the three categories it belongs to.
2. If the LLM answer is a **Wrong Answer**, output:  
   wrong
3. If the LLM answer is a **Blank Answer**, output:  
   blank
4. If the LLM answer is a **Refusal to Answer**, output:  
   refusal
5. If the classification is unclear, choose the category that best fits the content of the LLM answer.

Do not output explanations, justifications, or any additional text — only the words "wrong", "blank", or "refusal".

**LLM Answer:**  
{llm_answer}

**Reference Answer 1:**  
{reference_answer_1}

**Reference Answer 2:**  
{reference_answer_2}

**Output:**
"""
)

In [ ]:
def get_claude_answer_analysis(prompt_template, llm_answer, reference_answer_1, reference_answer_2):
    claude_messages = [
        {"role": "user", "content": prompt_template.format(llm_answer=llm_answer, reference_answer_1=reference_answer_1, reference_answer_2=reference_answer_2)},
    ]
    claude_ans = get_claude_response(claude_messages)
    return claude_ans

In [ ]:
def compute_analysis(results, llm_answers, correct_answers, key1, key2):
    res=[]
    for i in range(len(results)):
        if results[i] == "none":
            llm_answer = llm_answers[i][key1]
            reference_answer_1 = correct_answers[i][key2]
            reference_answer_2 = correct_answers[i]["answer_real"]
        
            res.append(get_claude_answer_analysis(analysis_prompt, llm_answer, reference_answer_1, reference_answer_2))
        
    return res

In [ ]:
Y1_zero1_analysis = compute_analysis(Y1_zero1, llm_answers1, correct_answers1, "Y1_redefinition_zero", "answer1")
Y1_cot1_analysis = compute_analysis(Y1_cot1, llm_answers1, correct_answers1, "Y1_redefinition_cot", "answer1")
Y1_few1_analysis = compute_analysis(Y1_few1, llm_answers1, correct_answers1, "Y1_redefinition_few", "answer1")

Y2_zero1_analysis = compute_analysis(Y2_zero1, llm_answers1, correct_answers1, "Y2_redefinition_zero", "answer2")
Y2_cot1_analysis = compute_analysis(Y2_cot1, llm_answers1, correct_answers1, "Y2_redefinition_cot", "answer2")
Y2_few1_analysis = compute_analysis(Y2_few1, llm_answers1, correct_answers1, "Y2_redefinition_few", "answer2")

Y3_zero1_analysis = compute_analysis(Y3_zero1, llm_answers1, correct_answers1, "Y3_redefinition_zero", "answer3")
Y3_cot1_analysis = compute_analysis(Y3_cot1, llm_answers1, correct_answers1, "Y3_redefinition_cot", "answer3")
Y3_few1_analysis = compute_analysis(Y3_few1, llm_answers1, correct_answers1, "Y3_redefinition_few", "answer3")

C1_zero1_analysis = compute_analysis(C1_zero1, llm_answers1, correct_answers1, "C1_redefinition_zero", "answerC1")
C1_cot1_analysis = compute_analysis(C1_cot1, llm_answers1, correct_answers1, "C1_redefinition_cot", "answerC1")
C1_few1_analysis = compute_analysis(C1_few1, llm_answers1, correct_answers1, "C1_redefinition_few", "answerC1")

C2_zero1_analysis = compute_analysis(C2_zero1, llm_answers1, correct_answers1, "C2_redefinition_zero", "answerC2")
C2_cot1_analysis = compute_analysis(C2_cot1, llm_answers1, correct_answers1, "C2_redefinition_cot", "answerC2")
C2_few1_analysis = compute_analysis(C2_few1, llm_answers1, correct_answers1, "C2_redefinition_few", "answerC2")

In [ ]:
Y1_zero2_analysis = compute_analysis(Y1_zero2, llm_answers2, correct_answers2, "Y1_redefinition_zero", "answer1")
Y1_cot2_analysis = compute_analysis(Y1_cot2, llm_answers2, correct_answers2, "Y1_redefinition_cot", "answer1")
Y1_few2_analysis = compute_analysis(Y1_few2, llm_answers2, correct_answers2, "Y1_redefinition_few", "answer1")

Y2_zero2_analysis = compute_analysis(Y2_zero2, llm_answers2, correct_answers2, "Y2_redefinition_zero", "answer2")
Y2_cot2_analysis = compute_analysis(Y2_cot2, llm_answers2, correct_answers2, "Y2_redefinition_cot", "answer2")
Y2_few2_analysis = compute_analysis(Y2_few2, llm_answers2, correct_answers2, "Y2_redefinition_few", "answer2")

Y3_zero2_analysis = compute_analysis(Y3_zero2, llm_answers2, correct_answers2, "Y3_redefinition_zero", "answer3")
Y3_cot2_analysis = compute_analysis(Y3_cot2, llm_answers2, correct_answers2, "Y3_redefinition_cot", "answer3")
Y3_few2_analysis = compute_analysis(Y3_few2, llm_answers2, correct_answers2, "Y3_redefinition_few", "answer3")

C1_zero2_analysis = compute_analysis(C1_zero2, llm_answers2, correct_answers2, "C1_redefinition_zero", "answerC1")
C1_cot2_analysis = compute_analysis(C1_cot2, llm_answers2, correct_answers2, "C1_redefinition_cot", "answerC1")
C1_few2_analysis = compute_analysis(C1_few2, llm_answers2, correct_answers2, "C1_redefinition_few", "answerC1")

C2_zero2_analysis = compute_analysis(C2_zero2, llm_answers2, correct_answers2, "C2_redefinition_zero", "answerC2")
C2_cot2_analysis = compute_analysis(C2_cot2, llm_answers2, correct_answers2, "C2_redefinition_cot", "answerC2")
C2_few2_analysis = compute_analysis(C2_few2, llm_answers2, correct_answers2, "C2_redefinition_few", "answerC2")


In [ ]:
Y1_zero3_analysis = compute_analysis(Y1_zero3, llm_answers3, correct_answers3, "Y1_redefinition_zero", "answer1")
Y1_cot3_analysis = compute_analysis(Y1_cot3, llm_answers3, correct_answers3, "Y1_redefinition_cot", "answer1")
Y1_few3_analysis = compute_analysis(Y1_few3, llm_answers3, correct_answers3, "Y1_redefinition_few", "answer1")

Y2_zero3_analysis = compute_analysis(Y2_zero3, llm_answers3, correct_answers3, "Y2_redefinition_zero", "answer2")
Y2_cot3_analysis = compute_analysis(Y2_cot3, llm_answers3, correct_answers3, "Y2_redefinition_cot", "answer2")
Y2_few3_analysis = compute_analysis(Y2_few3, llm_answers3, correct_answers3, "Y2_redefinition_few", "answer2")

Y3_zero3_analysis = compute_analysis(Y3_zero3, llm_answers3, correct_answers3, "Y3_redefinition_zero", "answer3")
Y3_cot3_analysis = compute_analysis(Y3_cot3, llm_answers3, correct_answers3, "Y3_redefinition_cot", "answer3")
Y3_few3_analysis = compute_analysis(Y3_few3, llm_answers3, correct_answers3, "Y3_redefinition_few", "answer3")

C1_zero3_analysis = compute_analysis(C1_zero3, llm_answers3, correct_answers3, "C1_redefinition_zero", "answerC1")
C1_cot3_analysis = compute_analysis(C1_cot3, llm_answers3, correct_answers3, "C1_redefinition_cot", "answerC1")
C1_few3_analysis = compute_analysis(C1_few3, llm_answers3, correct_answers3, "C1_redefinition_few", "answerC1")

C2_zero3_analysis = compute_analysis(C2_zero3, llm_answers3, correct_answers3, "C2_redefinition_zero", "answerC2")
C2_cot3_analysis = compute_analysis(C2_cot3, llm_answers3, correct_answers3, "C2_redefinition_cot", "answerC2")
C2_few3_analysis = compute_analysis(C2_few3, llm_answers3, correct_answers3, "C2_redefinition_few", "answerC2")


## **Multiple Choice**

In [ ]:
comparison_prompt_redefinition_mc = PromptTemplate(
    input_variables=["llm_answer", "reference_answer_1", "reference_answer_2"],
    template="""
You are tasked with comparing an answer provided by an LLM (the "LLM answer") to two reference answers: "Reference Answer 1" and "Reference Answer 2". Your job is to determine if the LLM answer matches the letter of either of the two reference answers (A, B, C, or D).

**Instructions:**
1. Compare the LLM answer carefully with "Reference Answer 1" and "Reference Answer 2".
2. If the LLM answer matches "Reference Answer 1", output:  
   first
3. If the LLM answer matches "Reference Answer 2", output:  
   second
4. If the LLM answer matches neither of the two, output:  
   none

Do not output explanations, justifications, or any additional text — only the words "first", "second", or "none".

**LLM Answer:**  
{llm_answer}

**Reference Answer 1:**  
{reference_answer_1}

**Reference Answer 2:**  
{reference_answer_2}

**Output:**
"""
)

In [ ]:
def get_claude_answer_redefinition_mc(prompt_template, llm_answer, reference_answer_1, reference_answer_2):
    claude_messages = [
        {"role": "user", "content": comparison_prompt_redefinition_mc.format(llm_answer=llm_answer, reference_answer_1=reference_answer_1, reference_answer_2=reference_answer_2)},
    ]
    claude_ans = get_claude_response(claude_messages)
    return claude_ans

In [ ]:
def compute_res_mc(llm_answers, correct_answers, key1, key2, key3):
    
    res=[]
    for i in range(len(llm_answers)):
        llm_answer = llm_answers[i][key1]
        reference_answer_1 = correct_answers[i][key2]
        reference_answer_2 = correct_answers[i][key3]
        
        res.append(get_claude_answer_redefinition_mc(comparison_prompt_redefinition_mc, llm_answer, reference_answer_1, reference_answer_2
        ))
        
    return res

In [ ]:
Y1_zero1_mc=compute_res_mc(llm_answers1, correct_answers1_mc, "Y1_redefinition_mc_zero", "answer1", "answer1_real")
Y1_cot1_mc=compute_res_mc(llm_answers1, correct_answers1_mc, "Y1_redefinition_mc_cot", "answer1", "answer1_real")
Y1_few1_mc=compute_res_mc(llm_answers1, correct_answers1_mc, "Y1_redefinition_mc_few", "answer1", "answer1_real")

Y2_zero1_mc=compute_res_mc(llm_answers1, correct_answers1_mc, "Y2_redefinition_mc_zero", "answer2", "answer2_real")
Y2_cot1_mc=compute_res_mc(llm_answers1, correct_answers1_mc, "Y2_redefinition_mc_cot", "answer2", "answer2_real")
Y2_few1_mc=compute_res_mc(llm_answers1, correct_answers1_mc, "Y2_redefinition_mc_few", "answer2", "answer2_real")

Y3_zero1_mc=compute_res_mc(llm_answers1, correct_answers1_mc, "Y3_redefinition_mc_zero", "answer3", "answer3_real")
Y3_cot1_mc=compute_res_mc(llm_answers1, correct_answers1_mc, "Y3_redefinition_mc_cot", "answer3", "answer3_real")
Y3_few1_mc=compute_res_mc(llm_answers1, correct_answers1_mc, "Y3_redefinition_mc_few", "answer3", "answer3_real")

C1_zero1_mc=compute_res_mc(llm_answers1, correct_answers1_mc, "C1_redefinition_mc_zero", "answerC1", "answerC1_real")
C1_cot1_mc=compute_res_mc(llm_answers1, correct_answers1_mc, "C1_redefinition_mc_cot", "answerC1", "answerC1_real")
C1_few1_mc=compute_res_mc(llm_answers1, correct_answers1_mc, "C1_redefinition_mc_few", "answerC1", "answerC1_real")

C2_zero1_mc=compute_res_mc(llm_answers1, correct_answers1_mc, "C2_redefinition_mc_zero", "answerC2", "answerC2_real")
C2_cot1_mc=compute_res_mc(llm_answers1, correct_answers1_mc, "C2_redefinition_mc_cot", "answerC2", "answerC2_real")
C2_few1_mc=compute_res_mc(llm_answers1, correct_answers1_mc, "C2_redefinition_mc_few", "answerC2", "answerC2_real")

In [ ]:
Y1_zero2_mc=compute_res_mc(llm_answers2, correct_answers2_mc, "Y1_redefinition_mc_zero", "answer1", "answer1_real")
Y1_cot2_mc=compute_res_mc(llm_answers2, correct_answers2_mc, "Y1_redefinition_mc_cot", "answer1", "answer1_real")
Y1_few2_mc=compute_res_mc(llm_answers2, correct_answers2_mc, "Y1_redefinition_mc_few", "answer1", "answer1_real")

Y2_zero2_mc=compute_res_mc(llm_answers2, correct_answers2_mc, "Y2_redefinition_mc_zero", "answer2", "answer2_real")
Y2_cot2_mc=compute_res_mc(llm_answers2, correct_answers2_mc, "Y2_redefinition_mc_cot", "answer2", "answer2_real")
Y2_few2_mc=compute_res_mc(llm_answers2, correct_answers2_mc, "Y2_redefinition_mc_few", "answer2", "answer2_real")

Y3_zero2_mc=compute_res_mc(llm_answers2, correct_answers2_mc, "Y3_redefinition_mc_zero", "answer3", "answer3_real")
Y3_cot2_mc=compute_res_mc(llm_answers2, correct_answers2_mc, "Y3_redefinition_mc_cot", "answer3", "answer3_real")
Y3_few2_mc=compute_res_mc(llm_answers2, correct_answers2_mc, "Y3_redefinition_mc_few", "answer3", "answer3_real")

C1_zero2_mc=compute_res_mc(llm_answers2, correct_answers2_mc, "C1_redefinition_mc_zero", "answerC1", "answerC1_real")
C1_cot2_mc=compute_res_mc(llm_answers2, correct_answers2_mc, "C1_redefinition_mc_cot", "answerC1", "answerC1_real")
C1_few2_mc=compute_res_mc(llm_answers2, correct_answers2_mc, "C1_redefinition_mc_few", "answerC1", "answerC1_real")

C2_zero2_mc=compute_res_mc(llm_answers2, correct_answers2_mc, "C2_redefinition_mc_zero", "answerC2", "answerC2_real")
C2_cot2_mc=compute_res_mc(llm_answers2, correct_answers2_mc, "C2_redefinition_mc_cot", "answerC2", "answerC2_real")
C2_few2_mc=compute_res_mc(llm_answers2, correct_answers2_mc, "C2_redefinition_mc_few", "answerC2", "answerC2_real")

In [ ]:
Y1_zero3_mc=compute_res_mc(llm_answers3, correct_answers3_mc, "Y1_redefinition_mc_zero", "answer1", "answer1_real")
Y1_cot3_mc=compute_res_mc(llm_answers3, correct_answers3_mc, "Y1_redefinition_mc_cot", "answer1", "answer1_real")
Y1_few3_mc=compute_res_mc(llm_answers3, correct_answers3_mc, "Y1_redefinition_mc_few", "answer1", "answer1_real")

Y2_zero3_mc=compute_res_mc(llm_answers3, correct_answers3_mc, "Y2_redefinition_mc_zero", "answer2", "answer2_real")
Y2_cot3_mc=compute_res_mc(llm_answers3, correct_answers3_mc, "Y2_redefinition_mc_cot", "answer2", "answer2_real")
Y2_few3_mc=compute_res_mc(llm_answers3, correct_answers3_mc, "Y2_redefinition_mc_few", "answer2", "answer2_real")

Y3_zero3_mc=compute_res_mc(llm_answers3, correct_answers3_mc, "Y3_redefinition_mc_zero", "answer3", "answer3_real")
Y3_cot3_mc=compute_res_mc(llm_answers3, correct_answers3_mc, "Y3_redefinition_mc_cot", "answer3", "answer3_real")
Y3_few3_mc=compute_res_mc(llm_answers3, correct_answers3_mc, "Y3_redefinition_mc_few", "answer3", "answer3_real")

C1_zero3_mc=compute_res_mc(llm_answers3, correct_answers3_mc, "C1_redefinition_mc_zero", "answerC1", "answerC1_real")
C1_cot3_mc=compute_res_mc(llm_answers3, correct_answers3_mc, "C1_redefinition_mc_cot", "answerC1", "answerC1_real")
C1_few3_mc=compute_res_mc(llm_answers3, correct_answers3_mc, "C1_redefinition_mc_few", "answerC1", "answerC1_real")

C2_zero3_mc=compute_res_mc(llm_answers3, correct_answers3_mc, "C2_redefinition_mc_zero", "answerC2", "answerC2_real")
C2_cot3_mc=compute_res_mc(llm_answers3, correct_answers3_mc, "C2_redefinition_mc_cot", "answerC2", "answerC2_real")
C2_few3_mc=compute_res_mc(llm_answers3, correct_answers3_mc, "C2_redefinition_mc_few", "answerC2", "answerC2_real")

In [ ]:
def get_claude_answer_analysis_mc(prompt_template, llm_answer, reference_answer_1, reference_answer_2):
    claude_messages = [
        {"role": "user", "content": prompt_template.format(llm_answer=llm_answer, reference_answer_1=reference_answer_1, reference_answer_2=reference_answer_2)},
    ]
    claude_ans = get_claude_response(claude_messages)
    return claude_ans

In [ ]:
def compute_analysis_mc(results, llm_answers, correct_answers, key1, key2, key3):
    
    res=[]
    for i in range(len(results)):
        if results[i] == "none":
            llm_answer = llm_answers[i][key1]
            reference_answer_1 = correct_answers[i][key2]
            reference_answer_2 = correct_answers[i][key3]
            res.append(get_claude_answer_analysis_mc(analysis_prompt, llm_answer, reference_answer_1, reference_answer_2))
        
    return res

In [ ]:
Y1_zero1_mc_analysis = compute_analysis_mc(Y1_zero1_mc, llm_answers1, correct_answers1_mc, "Y1_redefinition_mc_zero", "answer1", "answer1_real")
Y1_cot1_mc_analysis = compute_analysis_mc(Y1_cot1_mc, llm_answers1, correct_answers1_mc, "Y1_redefinition_mc_cot", "answer1", "answer1_real")
Y1_few1_mc_analysis = compute_analysis_mc(Y1_few1_mc, llm_answers1, correct_answers1_mc, "Y1_redefinition_mc_few", "answer1", "answer1_real")

Y2_zero1_mc_analysis = compute_analysis_mc(Y2_zero1_mc, llm_answers1, correct_answers1_mc, "Y2_redefinition_mc_zero", "answer2", "answer2_real")
Y2_cot1_mc_analysis = compute_analysis_mc(Y2_cot1_mc, llm_answers1, correct_answers1_mc, "Y2_redefinition_mc_cot", "answer2", "answer2_real")
Y2_few1_mc_analysis = compute_analysis_mc(Y2_few1_mc, llm_answers1, correct_answers1_mc, "Y2_redefinition_mc_few", "answer2", "answer2_real")

Y3_zero1_mc_analysis = compute_analysis_mc(Y3_zero1_mc, llm_answers1, correct_answers1_mc, "Y3_redefinition_mc_zero", "answer3", "answer3_real")
Y3_cot1_mc_analysis = compute_analysis_mc(Y3_cot1_mc, llm_answers1, correct_answers1_mc, "Y3_redefinition_mc_cot", "answer3", "answer3_real")
Y3_few1_mc_analysis = compute_analysis_mc(Y3_few1_mc, llm_answers1, correct_answers1_mc, "Y3_redefinition_mc_few", "answer3", "answer3_real")

C1_zero1_mc_analysis = compute_analysis_mc(C1_zero1_mc, llm_answers1, correct_answers1_mc, "C1_redefinition_mc_zero", "answerC1", "answerC1_real")
C1_cot1_mc_analysis = compute_analysis_mc(C1_cot1_mc, llm_answers1, correct_answers1_mc, "C1_redefinition_mc_cot", "answerC1", "answerC1_real")
C1_few1_mc_analysis = compute_analysis_mc(C1_few1_mc, llm_answers1, correct_answers1_mc, "C1_redefinition_mc_few", "answerC1", "answerC1_real")

C2_zero1_mc_analysis = compute_analysis_mc(C2_zero1_mc, llm_answers1, correct_answers1_mc, "C2_redefinition_mc_zero", "answerC2", "answerC2_real")
C2_cot1_mc_analysis = compute_analysis_mc(C2_cot1_mc, llm_answers1, correct_answers1_mc, "C2_redefinition_mc_cot", "answerC2", "answerC2_real")
C2_few1_mc_analysis = compute_analysis_mc(C2_few1_mc, llm_answers1, correct_answers1_mc, "C2_redefinition_mc_few", "answerC2", "answerC2_real")

In [ ]:
Y1_zero2_mc_analysis = compute_analysis_mc(Y1_zero2_mc, llm_answers2, correct_answers2_mc, "Y1_redefinition_mc_zero", "answer1", "answer1_real")
Y1_cot2_mc_analysis = compute_analysis_mc(Y1_cot2_mc, llm_answers2, correct_answers2_mc, "Y1_redefinition_mc_cot", "answer1", "answer1_real")
Y1_few2_mc_analysis = compute_analysis_mc(Y1_few2_mc, llm_answers2, correct_answers2_mc, "Y1_redefinition_mc_few", "answer1", "answer1_real")

Y2_zero2_mc_analysis = compute_analysis_mc(Y2_zero2_mc, llm_answers2, correct_answers2_mc, "Y2_redefinition_mc_zero", "answer2", "answer2_real")
Y2_cot2_mc_analysis = compute_analysis_mc(Y2_cot2_mc, llm_answers2, correct_answers2_mc, "Y2_redefinition_mc_cot", "answer2", "answer2_real")
Y2_few2_mc_analysis = compute_analysis_mc(Y2_few2_mc, llm_answers2, correct_answers2_mc, "Y2_redefinition_mc_few", "answer2", "answer2_real")

Y3_zero2_mc_analysis = compute_analysis_mc(Y3_zero2_mc, llm_answers2, correct_answers2_mc, "Y3_redefinition_mc_zero", "answer3", "answer3_real")
Y3_cot2_mc_analysis = compute_analysis_mc(Y3_cot2_mc, llm_answers2, correct_answers2_mc, "Y3_redefinition_mc_cot", "answer3", "answer3_real")
Y3_few2_mc_analysis = compute_analysis_mc(Y3_few2_mc, llm_answers2, correct_answers2_mc, "Y3_redefinition_mc_few", "answer3", "answer3_real")

C1_zero2_mc_analysis = compute_analysis_mc(C1_zero2_mc, llm_answers2, correct_answers2_mc, "C1_redefinition_mc_zero", "answerC1", "answerC1_real")
C1_cot2_mc_analysis = compute_analysis_mc(C1_cot2_mc, llm_answers2, correct_answers2_mc, "C1_redefinition_mc_cot", "answerC1", "answerC1_real")
C1_few2_mc_analysis = compute_analysis_mc(C1_few2_mc, llm_answers2, correct_answers2_mc, "C1_redefinition_mc_few", "answerC1", "answerC1_real")

C2_zero2_mc_analysis = compute_analysis_mc(C2_zero2_mc, llm_answers2, correct_answers2_mc, "C2_redefinition_mc_zero", "answerC2", "answerC2_real")
C2_cot2_mc_analysis = compute_analysis_mc(C2_cot2_mc, llm_answers2, correct_answers2_mc, "C2_redefinition_mc_cot", "answerC2", "answerC2_real")
C2_few2_mc_analysis = compute_analysis_mc(C2_few2_mc, llm_answers2, correct_answers2_mc, "C2_redefinition_mc_few", "answerC2", "answerC2_real")

In [ ]:
Y1_zero3_mc_analysis = compute_analysis_mc(Y1_zero3_mc, llm_answers3, correct_answers3_mc, "Y1_redefinition_mc_zero", "answer1", "answer1_real")
Y1_cot3_mc_analysis = compute_analysis_mc(Y1_cot3_mc, llm_answers3, correct_answers3_mc, "Y1_redefinition_mc_cot", "answer1", "answer1_real")
Y1_few3_mc_analysis = compute_analysis_mc(Y1_few3_mc, llm_answers3, correct_answers3_mc, "Y1_redefinition_mc_few", "answer1", "answer1_real")

Y2_zero3_mc_analysis = compute_analysis_mc(Y2_zero3_mc, llm_answers3, correct_answers3_mc, "Y2_redefinition_mc_zero", "answer2", "answer2_real")
Y2_cot3_mc_analysis = compute_analysis_mc(Y2_cot3_mc, llm_answers3, correct_answers3_mc, "Y2_redefinition_mc_cot", "answer2", "answer2_real")
Y2_few3_mc_analysis = compute_analysis_mc(Y2_few3_mc, llm_answers3, correct_answers3_mc, "Y2_redefinition_mc_few", "answer2", "answer2_real")

Y3_zero3_mc_analysis = compute_analysis_mc(Y3_zero3_mc, llm_answers3, correct_answers3_mc, "Y3_redefinition_mc_zero", "answer3", "answer3_real")
Y3_cot3_mc_analysis = compute_analysis_mc(Y3_cot3_mc, llm_answers3, correct_answers3_mc, "Y3_redefinition_mc_cot", "answer3", "answer3_real")
Y3_few3_mc_analysis = compute_analysis_mc(Y3_few3_mc, llm_answers3, correct_answers3_mc, "Y3_redefinition_mc_few", "answer3", "answer3_real")

C1_zero3_mc_analysis = compute_analysis_mc(C1_zero3_mc, llm_answers3, correct_answers3_mc, "C1_redefinition_mc_zero", "answerC1", "answerC1_real")
C1_cot3_mc_analysis = compute_analysis_mc(C1_cot3_mc, llm_answers3, correct_answers3_mc, "C1_redefinition_mc_cot", "answerC1", "answerC1_real")
C1_few3_mc_analysis = compute_analysis_mc(C1_few3_mc, llm_answers3, correct_answers3_mc, "C1_redefinition_mc_few", "answerC1", "answerC1_real")

C2_zero3_mc_analysis = compute_analysis_mc(C2_zero3_mc, llm_answers3, correct_answers3_mc, "C2_redefinition_mc_zero", "answerC2", "answerC2_real")
C2_cot3_mc_analysis = compute_analysis_mc(C2_cot3_mc, llm_answers3, correct_answers3_mc, "C2_redefinition_mc_cot", "answerC2", "answerC2_real")
C2_few3_mc_analysis = compute_analysis_mc(C2_few3_mc, llm_answers3, correct_answers3_mc, "C2_redefinition_mc_few", "answerC2", "answerC2_real")

# **Results**

Results for evaluating the model's performance on the redefinition of constants are displayed in the tables below. The resulting tables are exported in .txt files. 

In [ ]:
from tabulate import tabulate

def calculate_results_table(results, names, i):
    table_data = []
    results_dict = {} 

    for name, res in zip(names, results):
        green = 0
        red = 0
        gray = 0

        for item in res:
            if item == "first":
                green += 1
            elif item == "second":
                red += 1

        green = 100 * green / len(res)
        red = 100 * red / len(res)
        gray = abs(100 - green - red)

        table_data.append([name, round(green, 2), round(red, 2), round(gray, 2)])

        results_dict[name] = {"green": round(green, 2), "red": round(red, 2), "gray": round(gray, 2)}

    return results_dict, table_data


def calculate_results_table_analysis(results, names, i):
    table_data = []
    results_dict = {} 

    for name, res in zip(names, results):
        wrong = 0
        refusal = 0
        blank = 0

        for item in res:
            if item == "wrong":
                wrong += 1
            elif item == "refusal":
                refusal += 1

        if len(res)==0:
            wrong = 0
            refusal = 0
            blank = 0
        else:
            wrong = 100 * wrong / len(res)
            refusal = 100 * refusal / len(res)
            blank = abs(100 - wrong - refusal)

        table_data.append([name, round(wrong, 2), round(refusal, 2), round(blank, 2)])

        results_dict[name] = {"wrong": round(wrong, 2), "refusal": round(refusal, 2), "blank": round(blank, 2)}

    return results_dict, table_data  


def format_tables_side_by_side(table_a, table_b):
    """Format two tables side by side with spaces in between."""
    table_a_lines = table_a.split("\n")
    table_b_lines = table_b.split("\n")

    max_width_a = max(len(line) for line in table_a_lines)

    combined_lines = []
    for line_a, line_b in zip(table_a_lines, table_b_lines):
        combined_lines.append(line_a.ljust(max_width_a + 4) + line_b)

    return "\n".join(combined_lines)


def write_tables_to_file(file_name, results1, results2, results3, results1_mc, results2_mc, results3_mc, names, acc1, acc2, acc3):
    with open(file_name, "w") as file:
        _, table1 = calculate_results_table(results1, names, 1)
        _, table1_mc = calculate_results_table(results1_mc, names, 1)

        _, table2 = calculate_results_table(results2, names, 2)
        _, table2_mc = calculate_results_table(results2_mc, names, 2)

        _, table3 = calculate_results_table(results3, names, 3)
        _, table3_mc = calculate_results_table(results3_mc, names, 3)

        def format_and_combine_tables(table_a_data, table_b_data, level):
            table_a = tabulate(table_a_data, headers=[f"Level {level} FF", "Correct (%)", "Anchored (%)", "Wrong (%)"], tablefmt="grid")
            table_b = tabulate(table_b_data, headers=[f"Level {level} MC", "Correct (%)", "Anchored (%)", "Wrong (%)"], tablefmt="grid")
            return format_tables_side_by_side(table_a, table_b)

        file.write("### Level 1 ###\n\n")
        file.write(f"No Redefinition: {acc1}%\n")
        combined_1 = format_and_combine_tables(table1, table1_mc, 1)
        file.write(combined_1 + "\n\n")

        file.write("### Level 2 ###\n\n")
        file.write(f"No Redefinition: {acc2}%\n")

        combined_2 = format_and_combine_tables(table2, table2_mc, 2)
        file.write(combined_2 + "\n\n")

        file.write("### Level 3 ###\n\n")
        file.write(f"No Redefinition: {acc3}%\n")
        combined_3 = format_and_combine_tables(table3, table3_mc, 3)
        file.write(combined_3 + "\n\n")

        print(f"Results successfully written to {file_name}")


def write_tables_to_file_analysis(file_name, results1, results2, results3, results1_mc, results2_mc, results3_mc, names):
    with open(file_name, "w") as file:
        _, table1 = calculate_results_table_analysis(results1, names, 1)
        _, table1_mc = calculate_results_table_analysis(results1_mc, names, 1)

        _, table2 = calculate_results_table_analysis(results2, names, 2)
        _, table2_mc = calculate_results_table_analysis(results2_mc, names, 2)

        _, table3 = calculate_results_table_analysis(results3, names, 3)
        _, table3_mc = calculate_results_table_analysis(results3_mc, names, 3)

        def format_and_combine_tables(table_a_data, table_b_data, level):
            table_a = tabulate(table_a_data, headers=[f"Level {level} FF", "Wrong (%)", "Refusal (%)", "Blank (%)"], tablefmt="grid")
            table_b = tabulate(table_b_data, headers=[f"Level {level} MC", "Wrong (%)", "Refusal (%)", "Blank (%)"], tablefmt="grid")
            return format_tables_side_by_side(table_a, table_b)

        file.write("### Level 1 ###\n\n")
        combined_1 = format_and_combine_tables(table1, table1_mc, 1)
        file.write(combined_1 + "\n\n")

        file.write("### Level 2 ###\n\n")
        combined_2 = format_and_combine_tables(table2, table2_mc, 2)
        file.write(combined_2 + "\n\n")

        file.write("### Level 3 ###\n\n")
        combined_3 = format_and_combine_tables(table3, table3_mc, 3)
        file.write(combined_3 + "\n\n")

        print(f"Results successfully written to {file_name}")

In [ ]:
results1 = [
    Y1_zero1, Y1_cot1, Y1_few1, 
    Y2_zero1, Y2_cot1, Y2_few1, 
    Y3_zero1, Y3_cot1, Y3_few1, 
    C1_zero1, C1_cot1, C1_few1, 
    C2_zero1, C2_cot1, C2_few1
]

results2 = [
    Y1_zero2, Y1_cot2, Y1_few2, 
    Y2_zero2, Y2_cot2, Y2_few2, 
    Y3_zero2, Y3_cot2, Y3_few2, 
    C1_zero2, C1_cot2, C1_few2, 
    C2_zero2, C2_cot2, C2_few2
]

results3 = [
    Y1_zero3, Y1_cot3, Y1_few3, 
    Y2_zero3, Y2_cot3, Y2_few3, 
    Y3_zero3, Y3_cot3, Y3_few3, 
    C1_zero3, C1_cot3, C1_few3, 
    C2_zero3, C2_cot3, C2_few3
]

results1_mc = [
    Y1_zero1_mc, Y1_cot1_mc, Y1_few1_mc, 
    Y2_zero1_mc, Y2_cot1_mc, Y2_few1_mc, 
    Y3_zero1_mc, Y3_cot1_mc, Y3_few1_mc, 
    C1_zero1_mc, C1_cot1_mc, C1_few1_mc, 
    C2_zero1_mc, C2_cot1_mc, C2_few1_mc
]

results2_mc = [
    Y1_zero2_mc, Y1_cot2_mc, Y1_few2_mc, 
    Y2_zero2_mc, Y2_cot2_mc, Y2_few2_mc, 
    Y3_zero2_mc, Y3_cot2_mc, Y3_few2_mc, 
    C1_zero2_mc, C1_cot2_mc, C1_few2_mc, 
    C2_zero2_mc, C2_cot2_mc, C2_few2_mc
]

results3_mc = [
    Y1_zero3_mc, Y1_cot3_mc, Y1_few3_mc, 
    Y2_zero3_mc, Y2_cot3_mc, Y2_few3_mc, 
    Y3_zero3_mc, Y3_cot3_mc, Y3_few3_mc, 
    C1_zero3_mc, C1_cot3_mc, C1_few3_mc, 
    C2_zero3_mc, C2_cot3_mc, C2_few3_mc
]

names = [
    "Y1 - zero", "Y1 - cot", "Y1 - few", 
    "Y2 - zero", "Y2 - cot", "Y2 - few", 
    "Y3 - zero", "Y3 - cot", "Y3 - few", 
    "C1 - zero", "C1 - cot", "C1 - few", 
    "C2 - zero", "C2 - cot", "C2 - few"
]

In [ ]:
write_tables_to_file("results_tables.txt", results1, results2, results3, results1_mc, results2_mc, results3_mc, names, acc1, acc2, acc3)

In [ ]:
with open("results_tables.txt", "r") as file:
    content = file.read()
print(content)

In [ ]:
results1_analysis = [
    Y1_zero1_analysis, Y1_cot1_analysis, Y1_few1_analysis, 
    Y2_zero1_analysis, Y2_cot1_analysis, Y2_few1_analysis, 
    Y3_zero1_analysis, Y3_cot1_analysis, Y3_few1_analysis, 
    C1_zero1_analysis, C1_cot1_analysis, C1_few1_analysis, 
    C2_zero1_analysis, C2_cot1_analysis, C2_few1_analysis
]

results2_analysis = [
    Y1_zero2_analysis, Y1_cot2_analysis, Y1_few2_analysis, 
    Y2_zero2_analysis, Y2_cot2_analysis, Y2_few2_analysis, 
    Y3_zero2_analysis, Y3_cot2_analysis, Y3_few2_analysis, 
    C1_zero2_analysis, C1_cot2_analysis, C1_few2_analysis, 
    C2_zero2_analysis, C2_cot2_analysis, C2_few2_analysis
]

results3_analysis = [
    Y1_zero3_analysis, Y1_cot3_analysis, Y1_few3_analysis, 
    Y2_zero3_analysis, Y2_cot3_analysis, Y2_few3_analysis, 
    Y3_zero3_analysis, Y3_cot3_analysis, Y3_few3_analysis, 
    C1_zero3_analysis, C1_cot3_analysis, C1_few3_analysis, 
    C2_zero3_analysis, C2_cot3_analysis, C2_few3_analysis
]

results1_mc_analysis = [
    Y1_zero1_mc_analysis, Y1_cot1_mc_analysis, Y1_few1_mc_analysis, 
    Y2_zero1_mc_analysis, Y2_cot1_mc_analysis, Y2_few1_mc_analysis, 
    Y3_zero1_mc_analysis, Y3_cot1_mc_analysis, Y3_few1_mc_analysis, 
    C1_zero1_mc_analysis, C1_cot1_mc_analysis, C1_few1_mc_analysis, 
    C2_zero1_mc_analysis, C2_cot1_mc_analysis, C2_few1_mc_analysis
]

results2_mc_analysis = [
    Y1_zero2_mc_analysis, Y1_cot2_mc_analysis, Y1_few2_mc_analysis, 
    Y2_zero2_mc_analysis, Y2_cot2_mc_analysis, Y2_few2_mc_analysis, 
    Y3_zero2_mc_analysis, Y3_cot2_mc_analysis, Y3_few2_mc_analysis, 
    C1_zero2_mc_analysis, C1_cot2_mc_analysis, C1_few2_mc_analysis, 
    C2_zero2_mc_analysis, C2_cot2_mc_analysis, C2_few2_mc_analysis
]

results3_mc_analysis = [
    Y1_zero3_mc_analysis, Y1_cot3_mc_analysis, Y1_few3_mc_analysis, 
    Y2_zero3_mc_analysis, Y2_cot3_mc_analysis, Y2_few3_mc_analysis, 
    Y3_zero3_mc_analysis, Y3_cot3_mc_analysis, Y3_few3_mc_analysis, 
    C1_zero3_mc_analysis, C1_cot3_mc_analysis, C1_few3_mc_analysis, 
    C2_zero3_mc_analysis, C2_cot3_mc_analysis, C2_few3_mc_analysis
]

In [ ]:
write_tables_to_file_analysis("wrong_responses_analysis_tables.txt", results1_analysis, results2_analysis, results3_analysis, results1_mc_analysis, results2_mc_analysis, results3_mc_analysis, names)

In [ ]:
with open("wrong_responses_analysis_tables.txt", "r") as file:
    content = file.read()
print(content)

In [ ]:
res_dict1, _ = calculate_results_table(results1, names, 1)
res_dict2, _ = calculate_results_table(results2, names, 2)
res_dict3, _ = calculate_results_table(results3, names, 3)

res_dict1_mc, _ = calculate_results_table(results1_mc, names, 1)
res_dict2_mc, _ = calculate_results_table(results2_mc, names, 2)
res_dict3_mc, _ = calculate_results_table(results3_mc, names, 3)

In [ ]:
res_dict1_analysis, _ = calculate_results_table_analysis(results1_analysis, names, 1)
res_dict2_analysis, _ = calculate_results_table_analysis(results2_analysis, names, 2)
res_dict3_analysis, _ = calculate_results_table_analysis(results3_analysis, names, 3)

res_dict1_mc_analysis, _ = calculate_results_table_analysis(results1_mc_analysis, names, 1)
res_dict2_mc_analysis, _ = calculate_results_table_analysis(results2_mc_analysis, names, 2)
res_dict3_mc_analysis, _ = calculate_results_table_analysis(results3_mc_analysis, names, 3)

To visualize these results we used the following R scripts for generating plots for the model's evaluation and the Completely Wrong Responses analysis.

In [ ]:
r_script = f"""
library(ggplot2)
library(reshape2)
library(gridExtra)  

test1 <- data.frame(level = c("value1", "value2", "value3", "constant1", "constant2"),
                    NoR_C = c({acc1}, {acc1}, {acc1}, {acc1}, {acc1}),     
                    Z_C = c({res_dict1["Y1 - zero"]["green"]}, {res_dict1["Y2 - zero"]["green"]}, {res_dict1["Y3 - zero"]["green"]}, {res_dict1["C1 - zero"]["green"]}, {res_dict1["C2 - zero"]["green"]}), 
                    Z_In = c({res_dict1["Y1 - zero"]["red"]}, {res_dict1["Y2 - zero"]["red"]}, {res_dict1["Y3 - zero"]["red"]}, {res_dict1["C1 - zero"]["red"]}, {res_dict1["C2 - zero"]["red"]}),
                    C_C = c({res_dict1["Y1 - cot"]["green"]}, {res_dict1["Y2 - cot"]["green"]}, {res_dict1["Y3 - cot"]["green"]}, {res_dict1["C1 - cot"]["green"]}, {res_dict1["C2 - cot"]["green"]}), 
                    C_In = c({res_dict1["Y1 - cot"]["red"]}, {res_dict1["Y2 - cot"]["red"]}, {res_dict1["Y3 - cot"]["red"]}, {res_dict1["C1 - cot"]["red"]}, {res_dict1["C2 - cot"]["red"]}),
                    F_C = c({res_dict1["Y1 - few"]["green"]}, {res_dict1["Y2 - few"]["green"]}, {res_dict1["Y3 - few"]["green"]}, {res_dict1["C1 - few"]["green"]}, {res_dict1["C2 - few"]["green"]}), 
                    F_In = c({res_dict1["Y1 - few"]["red"]}, {res_dict1["Y2 - few"]["red"]}, {res_dict1["Y3 - few"]["red"]}, {res_dict1["C1 - few"]["red"]}, {res_dict1["C2 - few"]["red"]}))
                     
test2 <- data.frame(level = c("value1", "value2", "value3", "constant1", "constant2"),
                    NoR_C = c({acc2}, {acc2}, {acc2}, {acc2}, {acc2}),     
                    Z_C = c({res_dict2["Y1 - zero"]["green"]}, {res_dict2["Y2 - zero"]["green"]}, {res_dict2["Y3 - zero"]["green"]}, {res_dict2["C1 - zero"]["green"]}, {res_dict2["C2 - zero"]["green"]}), 
                    Z_In = c({res_dict2["Y1 - zero"]["red"]}, {res_dict2["Y2 - zero"]["red"]}, {res_dict2["Y3 - zero"]["red"]}, {res_dict2["C1 - zero"]["red"]}, {res_dict2["C2 - zero"]["red"]}),
                    C_C = c({res_dict2["Y1 - cot"]["green"]}, {res_dict2["Y2 - cot"]["green"]}, {res_dict2["Y3 - cot"]["green"]}, {res_dict2["C1 - cot"]["green"]}, {res_dict2["C2 - cot"]["green"]}), 
                    C_In = c({res_dict2["Y1 - cot"]["red"]}, {res_dict2["Y2 - cot"]["red"]}, {res_dict2["Y3 - cot"]["red"]}, {res_dict2["C1 - cot"]["red"]}, {res_dict2["C2 - cot"]["red"]}),
                    F_C = c({res_dict2["Y1 - few"]["green"]}, {res_dict2["Y2 - few"]["green"]}, {res_dict2["Y3 - few"]["green"]}, {res_dict2["C1 - few"]["green"]}, {res_dict2["C2 - few"]["green"]}), 
                    F_In = c({res_dict2["Y1 - few"]["red"]}, {res_dict2["Y2 - few"]["red"]}, {res_dict2["Y3 - few"]["red"]}, {res_dict2["C1 - few"]["red"]}, {res_dict2["C2 - few"]["red"]}))

test3 <- data.frame(level = c("value1", "value2", "value3", "constant1", "constant2"),
                    NoR_C = c({acc3}, {acc3}, {acc3}, {acc3}, {acc3}),     
                    Z_C = c({res_dict3["Y1 - zero"]["green"]}, {res_dict3["Y2 - zero"]["green"]}, {res_dict3["Y3 - zero"]["green"]}, {res_dict3["C1 - zero"]["green"]}, {res_dict3["C2 - zero"]["green"]}), 
                    Z_In = c({res_dict3["Y1 - zero"]["red"]}, {res_dict3["Y2 - zero"]["red"]}, {res_dict3["Y3 - zero"]["red"]}, {res_dict3["C1 - zero"]["red"]}, {res_dict3["C2 - zero"]["red"]}),
                    C_C = c({res_dict3["Y1 - cot"]["green"]}, {res_dict3["Y2 - cot"]["green"]}, {res_dict3["Y3 - cot"]["green"]}, {res_dict3["C1 - cot"]["green"]}, {res_dict3["C2 - cot"]["green"]}), 
                    C_In = c({res_dict3["Y1 - cot"]["red"]}, {res_dict3["Y2 - cot"]["red"]}, {res_dict3["Y3 - cot"]["red"]}, {res_dict3["C1 - cot"]["red"]}, {res_dict3["C2 - cot"]["red"]}),
                    F_C = c({res_dict3["Y1 - few"]["green"]}, {res_dict3["Y2 - few"]["green"]}, {res_dict3["Y3 - few"]["green"]}, {res_dict3["C1 - few"]["green"]}, {res_dict3["C2 - few"]["green"]}), 
                    F_In = c({res_dict3["Y1 - few"]["red"]}, {res_dict3["Y2 - few"]["red"]}, {res_dict3["Y3 - few"]["red"]}, {res_dict3["C1 - few"]["red"]}, {res_dict3["C2 - few"]["red"]}))

test1$level <- factor(test1$level, levels = c("value1", "value2", "value3", "constant1", "constant2"))
test2$level <- factor(test2$level, levels = c("value1", "value2", "value3", "constant1", "constant2"))
test3$level <- factor(test3$level, levels = c("value1", "value2", "value3", "constant1", "constant2"))

generate_plot <- function(test_data, plot_title) {{
  test_data$level <- factor(test_data$level, levels = c("value1", "value2", "value3", "constant1", "constant2"))
  
  melted_data <- melt(test_data, "level")
  melted_data$cat <- ''
  melted_data[melted_data$variable == 'NoR_C',]$cat <- "No Redef."
  melted_data[melted_data$variable %in% c('Z_C', 'Z_In'),]$cat <- "Zero-Shot"
  melted_data[melted_data$variable %in% c('C_C', 'C_In'),]$cat <- "CoT"
  melted_data[melted_data$variable %in% c('F_C', 'F_In'),]$cat <- "Few-Shot"
  
  melted_data$results <- ifelse(melted_data$variable == "NoR_C", "NoR_C_correct", 
                                ifelse(grepl("_C$", melted_data$variable), "correct", "correct in the real world"))
  
  aggregated_data <- aggregate(value ~ level + cat, melted_data, sum)
  aggregated_data$other <- 100 - aggregated_data$value
  
  other_rows <- data.frame(level = aggregated_data$level,
                           variable = NA,
                           value = aggregated_data$other,
                           cat = aggregated_data$cat,
                           results = "incorrect or no answer")
  
  melted_data <- rbind(melted_data, other_rows)
  melted_data$results <- factor(melted_data$results, levels = c("incorrect or no answer", "correct in the real world", "correct", "NoR_C_correct"))
  melted_data$cat <- factor(melted_data$cat, levels = c("No Redef.", "Zero-Shot", "CoT", "Few-Shot"))
  
  plot <- ggplot(melted_data, aes(x = cat, y = value, fill = results)) + 
    geom_bar(stat = 'identity', position = 'stack') + 
    scale_fill_manual(values = c("NoR_C_correct" = "deepskyblue3",
                                 "correct" = "mediumseagreen", 
                                 "correct in the real world" = "lightcoral", 
                                 "incorrect or no answer" = "gray")) + 
    facet_grid(~ level) +
    labs(x = "", y = "", title = plot_title) +
    theme_minimal() + 
    theme(legend.position = "none",
          plot.title = element_text(hjust = 0.5),
          axis.text.x = element_text(angle = 45, hjust = 1))
  
  return(plot)
}}

# Generate the plots
plot1 <- generate_plot(test1, "Level 1")
plot2 <- generate_plot(test2, "Level 2")
plot3 <- generate_plot(test3, "Level 3")

# Arrange the plots in a grid
grid.arrange(plot1, plot2, plot3, ncol = 3)


#-------------------------- Multiple Choice --------------------------------


test1 <- data.frame(level = c("value1", "value2", "value3", "constant1", "constant2"),
                       NoR_C = c({acc1}, {acc1}, {acc1}, {acc1}, {acc1}),     
                       Z_C = c({res_dict1_mc["Y1 - zero"]["green"]}, {res_dict1_mc["Y2 - zero"]["green"]}, {res_dict1_mc["Y3 - zero"]["green"]}, {res_dict1_mc["C1 - zero"]["green"]}, {res_dict1_mc["C2 - zero"]["green"]}), 
                       Z_In = c({res_dict1_mc["Y1 - zero"]["red"]}, {res_dict1_mc["Y2 - zero"]["red"]}, {res_dict1_mc["Y3 - zero"]["red"]}, {res_dict1_mc["C1 - zero"]["red"]}, {res_dict1_mc["C2 - zero"]["red"]}),
                       C_C = c({res_dict1_mc["Y1 - cot"]["green"]}, {res_dict1_mc["Y2 - cot"]["green"]}, {res_dict1_mc["Y3 - cot"]["green"]}, {res_dict1_mc["C1 - cot"]["green"]}, {res_dict1_mc["C2 - cot"]["green"]}), 
                       C_In = c({res_dict1_mc["Y1 - cot"]["red"]}, {res_dict1_mc["Y2 - cot"]["red"]}, {res_dict1_mc["Y3 - cot"]["red"]}, {res_dict1_mc["C1 - cot"]["red"]}, {res_dict1_mc["C2 - cot"]["red"]}),
                       F_C = c({res_dict1_mc["Y1 - few"]["green"]}, {res_dict1_mc["Y2 - few"]["green"]}, {res_dict1_mc["Y3 - few"]["green"]}, {res_dict1_mc["C1 - few"]["green"]}, {res_dict1_mc["C2 - few"]["green"]}), 
                       F_In = c({res_dict1_mc["Y1 - few"]["red"]}, {res_dict1_mc["Y2 - few"]["red"]}, {res_dict1_mc["Y3 - few"]["red"]}, {res_dict1_mc["C1 - few"]["red"]}, {res_dict1_mc["C2 - few"]["red"]}))

test2 <- data.frame(level = c("value1", "value2", "value3", "constant1", "constant2"),
                    NoR_C = c({acc2}, {acc2}, {acc2}, {acc2}, {acc2}),     
                    Z_C = c({res_dict2_mc["Y1 - zero"]["green"]}, {res_dict2_mc["Y2 - zero"]["green"]}, {res_dict2_mc["Y3 - zero"]["green"]}, {res_dict2_mc["C1 - zero"]["green"]}, {res_dict2_mc["C2 - zero"]["green"]}), 
                    Z_In = c({res_dict2_mc["Y1 - zero"]["red"]}, {res_dict2_mc["Y2 - zero"]["red"]}, {res_dict2_mc["Y3 - zero"]["red"]}, {res_dict2_mc["C1 - zero"]["red"]}, {res_dict2_mc["C2 - zero"]["red"]}),
                    C_C = c({res_dict2_mc["Y1 - cot"]["green"]}, {res_dict2_mc["Y2 - cot"]["green"]}, {res_dict2_mc["Y3 - cot"]["green"]}, {res_dict2_mc["C1 - cot"]["green"]}, {res_dict2_mc["C2 - cot"]["green"]}), 
                    C_In = c({res_dict2_mc["Y1 - cot"]["red"]}, {res_dict2_mc["Y2 - cot"]["red"]}, {res_dict2_mc["Y3 - cot"]["red"]}, {res_dict2_mc["C1 - cot"]["red"]}, {res_dict2_mc["C2 - cot"]["red"]}),
                    F_C = c({res_dict2_mc["Y1 - few"]["green"]}, {res_dict2_mc["Y2 - few"]["green"]}, {res_dict2_mc["Y3 - few"]["green"]}, {res_dict2_mc["C1 - few"]["green"]}, {res_dict2_mc["C2 - few"]["green"]}), 
                    F_In = c({res_dict2_mc["Y1 - few"]["red"]}, {res_dict2_mc["Y2 - few"]["red"]}, {res_dict2_mc["Y3 - few"]["red"]}, {res_dict2_mc["C1 - few"]["red"]}, {res_dict2_mc["C2 - few"]["red"]}))

test3 <- data.frame(level = c("value1", "value2", "value3", "constant1", "constant2"),
                    NoR_C = c({acc3}, {acc3}, {acc3}, {acc3}, {acc3}),     
                    Z_C = c({res_dict3_mc["Y1 - zero"]["green"]}, {res_dict3_mc["Y2 - zero"]["green"]}, {res_dict3_mc["Y3 - zero"]["green"]}, {res_dict3_mc["C1 - zero"]["green"]}, {res_dict3_mc["C2 - zero"]["green"]}), 
                    Z_In = c({res_dict3_mc["Y1 - zero"]["red"]}, {res_dict3_mc["Y2 - zero"]["red"]}, {res_dict3_mc["Y3 - zero"]["red"]}, {res_dict3_mc["C1 - zero"]["red"]}, {res_dict3_mc["C2 - zero"]["red"]}),
                    C_C = c({res_dict3_mc["Y1 - cot"]["green"]}, {res_dict3_mc["Y2 - cot"]["green"]}, {res_dict3_mc["Y3 - cot"]["green"]}, {res_dict3_mc["C1 - cot"]["green"]}, {res_dict3_mc["C2 - cot"]["green"]}), 
                    C_In = c({res_dict3_mc["Y1 - cot"]["red"]}, {res_dict3_mc["Y2 - cot"]["red"]}, {res_dict3_mc["Y3 - cot"]["red"]}, {res_dict3_mc["C1 - cot"]["red"]}, {res_dict3_mc["C2 - cot"]["red"]}),
                    F_C = c({res_dict3_mc["Y1 - few"]["green"]}, {res_dict3_mc["Y2 - few"]["green"]}, {res_dict3_mc["Y3 - few"]["green"]}, {res_dict3_mc["C1 - few"]["green"]}, {res_dict3_mc["C2 - few"]["green"]}), 
                    F_In = c({res_dict3_mc["Y1 - few"]["red"]}, {res_dict3_mc["Y2 - few"]["red"]}, {res_dict3_mc["Y3 - few"]["red"]}, {res_dict3_mc["C1 - few"]["red"]}, {res_dict3_mc["C2 - few"]["red"]}))

test1$level <- factor(test1$level, levels = c("value1", "value2", "value3", "constant1", "constant2"))
test2$level <- factor(test2$level, levels = c("value1", "value2", "value3", "constant1", "constant2"))
test3$level <- factor(test3$level, levels = c("value1", "value2", "value3", "constant1", "constant2"))

plot1 <- generate_plot(test1, "Level 1")
plot2 <- generate_plot(test2, "Level 2")
plot3 <- generate_plot(test3, "Level 3")

grid.arrange(plot1, plot2, plot3, ncol = 3)

"""

In [ ]:
with open("results_plots.r", "w") as file:
    file.write(r_script)

print("R script saved as 'results_plots.r'")

In [ ]:
r_script2 = f"""
library(ggplot2)
library(reshape2)
library(gridExtra)  

test1 <- data.frame(level = c("value1", "value2", "value3", "constant1", "constant2"),
                       Z_C = c({res_dict1_analysis["Y1 - zero"]["wrong"]}, {res_dict1_analysis["Y2 - zero"]["wrong"]}, {res_dict1_analysis["Y3 - zero"]["wrong"]}, {res_dict1_analysis["C1 - zero"]["wrong"]}, {res_dict1_analysis["C2 - zero"]["wrong"]}), 
                       Z_In = c({res_dict1_analysis["Y1 - zero"]["refusal"]}, {res_dict1_analysis["Y2 - zero"]["refusal"]}, {res_dict1_analysis["Y3 - zero"]["refusal"]}, {res_dict1_analysis["C1 - zero"]["refusal"]}, {res_dict1_analysis["C2 - zero"]["refusal"]}),
                       C_C = c({res_dict1_analysis["Y1 - cot"]["wrong"]}, {res_dict1_analysis["Y2 - cot"]["wrong"]}, {res_dict1_analysis["Y3 - cot"]["wrong"]}, {res_dict1_analysis["C1 - cot"]["wrong"]}, {res_dict1_analysis["C2 - cot"]["wrong"]}), 
                       C_In = c({res_dict1_analysis["Y1 - cot"]["refusal"]}, {res_dict1_analysis["Y2 - cot"]["refusal"]}, {res_dict1_analysis["Y3 - cot"]["refusal"]}, {res_dict1_analysis["C1 - cot"]["refusal"]}, {res_dict1_analysis["C2 - cot"]["refusal"]}),
                       F_C = c({res_dict1_analysis["Y1 - few"]["wrong"]}, {res_dict1_analysis["Y2 - few"]["wrong"]}, {res_dict1_analysis["Y3 - few"]["wrong"]}, {res_dict1_analysis["C1 - few"]["wrong"]}, {res_dict1_analysis["C2 - few"]["wrong"]}), 
                       F_In = c({res_dict1_analysis["Y1 - few"]["refusal"]}, {res_dict1_analysis["Y2 - few"]["refusal"]}, {res_dict1_analysis["Y3 - few"]["refusal"]}, {res_dict1_analysis["C1 - few"]["refusal"]}, {res_dict1_analysis["C2 - few"]["refusal"]}))

test2 <- data.frame(level = c("value1", "value2", "value3", "constant1", "constant2"),
                    Z_C = c({res_dict2_analysis["Y1 - zero"]["wrong"]}, {res_dict2_analysis["Y2 - zero"]["wrong"]}, {res_dict2_analysis["Y3 - zero"]["wrong"]}, {res_dict2_analysis["C1 - zero"]["wrong"]}, {res_dict2_analysis["C2 - zero"]["wrong"]}), 
                    Z_In = c({res_dict2_analysis["Y1 - zero"]["refusal"]}, {res_dict2_analysis["Y2 - zero"]["refusal"]}, {res_dict2_analysis["Y3 - zero"]["refusal"]}, {res_dict2_analysis["C1 - zero"]["refusal"]}, {res_dict2_analysis["C2 - zero"]["refusal"]}),
                    C_C = c({res_dict2_analysis["Y1 - cot"]["wrong"]}, {res_dict2_analysis["Y2 - cot"]["wrong"]}, {res_dict2_analysis["Y3 - cot"]["wrong"]}, {res_dict2_analysis["C1 - cot"]["wrong"]}, {res_dict2_analysis["C2 - cot"]["wrong"]}), 
                    C_In = c({res_dict2_analysis["Y1 - cot"]["refusal"]}, {res_dict2_analysis["Y2 - cot"]["refusal"]}, {res_dict2_analysis["Y3 - cot"]["refusal"]}, {res_dict2_analysis["C1 - cot"]["refusal"]}, {res_dict2_analysis["C2 - cot"]["refusal"]}),
                    F_C = c({res_dict2_analysis["Y1 - few"]["wrong"]}, {res_dict2_analysis["Y2 - few"]["wrong"]}, {res_dict2_analysis["Y3 - few"]["wrong"]}, {res_dict2_analysis["C1 - few"]["wrong"]}, {res_dict2_analysis["C2 - few"]["wrong"]}), 
                    F_In = c({res_dict2_analysis["Y1 - few"]["refusal"]}, {res_dict2_analysis["Y2 - few"]["refusal"]}, {res_dict2_analysis["Y3 - few"]["refusal"]}, {res_dict2_analysis["C1 - few"]["refusal"]}, {res_dict2_analysis["C2 - few"]["refusal"]}))

test3 <- data.frame(level = c("value1", "value2", "value3", "constant1", "constant2"),
                    Z_C = c({res_dict3_analysis["Y1 - zero"]["wrong"]}, {res_dict3_analysis["Y2 - zero"]["wrong"]}, {res_dict3_analysis["Y3 - zero"]["wrong"]}, {res_dict3_analysis["C1 - zero"]["wrong"]}, {res_dict3_analysis["C2 - zero"]["wrong"]}), 
                    Z_In = c({res_dict3_analysis["Y1 - zero"]["refusal"]}, {res_dict3_analysis["Y2 - zero"]["refusal"]}, {res_dict3_analysis["Y3 - zero"]["refusal"]}, {res_dict3_analysis["C1 - zero"]["refusal"]}, {res_dict3_analysis["C2 - zero"]["refusal"]}),
                    C_C = c({res_dict3_analysis["Y1 - cot"]["wrong"]}, {res_dict3_analysis["Y2 - cot"]["wrong"]}, {res_dict3_analysis["Y3 - cot"]["wrong"]}, {res_dict3_analysis["C1 - cot"]["wrong"]}, {res_dict3_analysis["C2 - cot"]["wrong"]}), 
                    C_In = c({res_dict3_analysis["Y1 - cot"]["refusal"]}, {res_dict3_analysis["Y2 - cot"]["refusal"]}, {res_dict3_analysis["Y3 - cot"]["refusal"]}, {res_dict3_analysis["C1 - cot"]["refusal"]}, {res_dict3_analysis["C2 - cot"]["refusal"]}),
                    F_C = c({res_dict3_analysis["Y1 - few"]["wrong"]}, {res_dict3_analysis["Y2 - few"]["wrong"]}, {res_dict3_analysis["Y3 - few"]["wrong"]}, {res_dict3_analysis["C1 - few"]["wrong"]}, {res_dict3_analysis["C2 - few"]["wrong"]}), 
                    F_In = c({res_dict3_analysis["Y1 - few"]["refusal"]}, {res_dict3_analysis["Y2 - few"]["refusal"]}, {res_dict3_analysis["Y3 - few"]["refusal"]}, {res_dict3_analysis["C1 - few"]["refusal"]}, {res_dict3_analysis["C2 - few"]["refusal"]}))


test1$level <- factor(test1$level, levels = c("value1", "value2", "value3", "constant1", "constant2"))
test2$level <- factor(test2$level, levels = c("value1", "value2", "value3", "constant1", "constant2"))
test3$level <- factor(test3$level, levels = c("value1", "value2", "value3", "constant1", "constant2"))

# Function to process and plot data for each test
generate_plot <- function(test_data, title) {{
  melted <- melt(test_data, "level")
  melted$cat <- ''
  melted[melted$variable %in% c('Z_C', 'Z_In'),]$cat <- "Zero-Shot"
  melted[melted$variable %in% c('C_C', 'C_In'),]$cat <- "CoT"
  melted[melted$variable %in% c('F_C', 'F_In'),]$cat <- "Few-Shot"
  
  melted$results <- ifelse(grepl("_C$", melted$variable), "correct", "correct in the real world")
  
  aggregated <- aggregate(value ~ level + cat, melted, sum)
  aggregated$other <- 100 - aggregated$value
  other_rows <- data.frame(level = aggregated$level,
                           variable = NA,
                           value = aggregated$other,
                           cat = aggregated$cat,
                           results = "incorrect or no answer")
  melted <- rbind(melted, other_rows)
  
  melted$results <- factor(melted$results, levels = c("incorrect or no answer", "correct in the real world", "correct"))
  melted$cat <- factor(melted$cat, levels = c("Zero-Shot", "CoT", "Few-Shot"))
  
  plot <- ggplot(melted, aes(x = cat, y = value, fill = results)) + 
    geom_bar(stat = 'identity', position = 'stack') + 
    scale_fill_manual(values = c("correct" = "lightblue3", 
                                 "correct in the real world" = "plum", 
                                 "incorrect or no answer" = "gray90")) + 
    facet_grid(~ level) +
    labs(x = "", y = "", title = title) +
    theme_minimal() + 
    theme(legend.position = "none",
          plot.title = element_text(hjust = 0.5),
          axis.text.x = element_text(angle = 45, hjust = 1))
  
  return(plot)
}}

# Generate plots
plot1 <- generate_plot(test1, "Level 1")
plot2 <- generate_plot(test2, "Level 2")
plot3 <- generate_plot(test3, "Level 3")

# Combine and display the plots
grid.arrange(plot1, plot2, plot3, ncol = 3)


#-------------------------- Multiple Choice --------------------------------


test1 <- data.frame(level = c("value1", "value2", "value3", "constant1", "constant2"),
                       Z_C = c({res_dict1_mc_analysis["Y1 - zero"]["wrong"]}, {res_dict1_mc_analysis["Y2 - zero"]["wrong"]}, {res_dict1_mc_analysis["Y3 - zero"]["wrong"]}, {res_dict1_mc_analysis["C1 - zero"]["wrong"]}, {res_dict1_mc_analysis["C2 - zero"]["wrong"]}), 
                       Z_In = c({res_dict1_mc_analysis["Y1 - zero"]["refusal"]}, {res_dict1_mc_analysis["Y2 - zero"]["refusal"]}, {res_dict1_mc_analysis["Y3 - zero"]["refusal"]}, {res_dict1_mc_analysis["C1 - zero"]["refusal"]}, {res_dict1_mc_analysis["C2 - zero"]["refusal"]}),
                       C_C = c({res_dict1_mc_analysis["Y1 - cot"]["wrong"]}, {res_dict1_mc_analysis["Y2 - cot"]["wrong"]}, {res_dict1_mc_analysis["Y3 - cot"]["wrong"]}, {res_dict1_mc_analysis["C1 - cot"]["wrong"]}, {res_dict1_mc_analysis["C2 - cot"]["wrong"]}), 
                       C_In = c({res_dict1_mc_analysis["Y1 - cot"]["refusal"]}, {res_dict1_mc_analysis["Y2 - cot"]["refusal"]}, {res_dict1_mc_analysis["Y3 - cot"]["refusal"]}, {res_dict1_mc_analysis["C1 - cot"]["refusal"]}, {res_dict1_mc_analysis["C2 - cot"]["refusal"]}),
                       F_C = c({res_dict1_mc_analysis["Y1 - few"]["wrong"]}, {res_dict1_mc_analysis["Y2 - few"]["wrong"]}, {res_dict1_mc_analysis["Y3 - few"]["wrong"]}, {res_dict1_mc_analysis["C1 - few"]["wrong"]}, {res_dict1_mc_analysis["C2 - few"]["wrong"]}), 
                       F_In = c({res_dict1_mc_analysis["Y1 - few"]["refusal"]}, {res_dict1_mc_analysis["Y2 - few"]["refusal"]}, {res_dict1_mc_analysis["Y3 - few"]["refusal"]}, {res_dict1_mc_analysis["C1 - few"]["refusal"]}, {res_dict1_mc_analysis["C2 - few"]["refusal"]}))

test2 <- data.frame(level = c("value1", "value2", "value3", "constant1", "constant2"),
                    Z_C = c({res_dict2_mc_analysis["Y1 - zero"]["wrong"]}, {res_dict2_mc_analysis["Y2 - zero"]["wrong"]}, {res_dict2_mc_analysis["Y3 - zero"]["wrong"]}, {res_dict2_mc_analysis["C1 - zero"]["wrong"]}, {res_dict2_mc_analysis["C2 - zero"]["wrong"]}), 
                    Z_In = c({res_dict2_mc_analysis["Y1 - zero"]["refusal"]}, {res_dict2_mc_analysis["Y2 - zero"]["refusal"]}, {res_dict2_mc_analysis["Y3 - zero"]["refusal"]}, {res_dict2_mc_analysis["C1 - zero"]["refusal"]}, {res_dict2_mc_analysis["C2 - zero"]["refusal"]}),
                    C_C = c({res_dict2_mc_analysis["Y1 - cot"]["wrong"]}, {res_dict2_mc_analysis["Y2 - cot"]["wrong"]}, {res_dict2_mc_analysis["Y3 - cot"]["wrong"]}, {res_dict2_mc_analysis["C1 - cot"]["wrong"]}, {res_dict2_mc_analysis["C2 - cot"]["wrong"]}), 
                    C_In = c({res_dict2_mc_analysis["Y1 - cot"]["refusal"]}, {res_dict2_mc_analysis["Y2 - cot"]["refusal"]}, {res_dict2_mc_analysis["Y3 - cot"]["refusal"]}, {res_dict2_mc_analysis["C1 - cot"]["refusal"]}, {res_dict2_mc_analysis["C2 - cot"]["refusal"]}),
                    F_C = c({res_dict2_mc_analysis["Y1 - few"]["wrong"]}, {res_dict2_mc_analysis["Y2 - few"]["wrong"]}, {res_dict2_mc_analysis["Y3 - few"]["wrong"]}, {res_dict2_mc_analysis["C1 - few"]["wrong"]}, {res_dict2_mc_analysis["C2 - few"]["wrong"]}), 
                    F_In = c({res_dict2_mc_analysis["Y1 - few"]["refusal"]}, {res_dict2_mc_analysis["Y2 - few"]["refusal"]}, {res_dict2_mc_analysis["Y3 - few"]["refusal"]}, {res_dict2_mc_analysis["C1 - few"]["refusal"]}, {res_dict2_mc_analysis["C2 - few"]["refusal"]}))

test3 <- data.frame(level = c("value1", "value2", "value3", "constant1", "constant2"),
                    Z_C = c({res_dict3_mc_analysis["Y1 - zero"]["wrong"]}, {res_dict3_mc_analysis["Y2 - zero"]["wrong"]}, {res_dict3_mc_analysis["Y3 - zero"]["wrong"]}, {res_dict3_mc_analysis["C1 - zero"]["wrong"]}, {res_dict3_mc_analysis["C2 - zero"]["wrong"]}), 
                    Z_In = c({res_dict3_mc_analysis["Y1 - zero"]["refusal"]}, {res_dict3_mc_analysis["Y2 - zero"]["refusal"]}, {res_dict3_mc_analysis["Y3 - zero"]["refusal"]}, {res_dict3_mc_analysis["C1 - zero"]["refusal"]}, {res_dict3_mc_analysis["C2 - zero"]["refusal"]}),
                    C_C = c({res_dict3_mc_analysis["Y1 - cot"]["wrong"]}, {res_dict3_mc_analysis["Y2 - cot"]["wrong"]}, {res_dict3_mc_analysis["Y3 - cot"]["wrong"]}, {res_dict3_mc_analysis["C1 - cot"]["wrong"]}, {res_dict3_mc_analysis["C2 - cot"]["wrong"]}), 
                    C_In = c({res_dict3_mc_analysis["Y1 - cot"]["refusal"]}, {res_dict3_mc_analysis["Y2 - cot"]["refusal"]}, {res_dict3_mc_analysis["Y3 - cot"]["refusal"]}, {res_dict3_mc_analysis["C1 - cot"]["refusal"]}, {res_dict3_mc_analysis["C2 - cot"]["refusal"]}),
                    F_C = c({res_dict3_mc_analysis["Y1 - few"]["wrong"]}, {res_dict3_mc_analysis["Y2 - few"]["wrong"]}, {res_dict3_mc_analysis["Y3 - few"]["wrong"]}, {res_dict3_mc_analysis["C1 - few"]["wrong"]}, {res_dict3_mc_analysis["C2 - few"]["wrong"]}), 
                    F_In = c({res_dict3_mc_analysis["Y1 - few"]["refusal"]}, {res_dict3_mc_analysis["Y2 - few"]["refusal"]}, {res_dict3_mc_analysis["Y3 - few"]["refusal"]}, {res_dict3_mc_analysis["C1 - few"]["refusal"]}, {res_dict3_mc_analysis["C2 - few"]["refusal"]}))



test1$level <- factor(test1$level, levels = c("value1", "value2", "value3", "constant1", "constant2"))
test2$level <- factor(test2$level, levels = c("value1", "value2", "value3", "constant1", "constant2"))
test3$level <- factor(test3$level, levels = c("value1", "value2", "value3", "constant1", "constant2"))

# Generate plots
plot1 <- generate_plot(test1, "Level 1")
plot2 <- generate_plot(test2, "Level 2")
plot3 <- generate_plot(test3, "Level 3")

# Combine and display the plots
grid.arrange(plot1, plot2, plot3, ncol = 3)

"""

In [ ]:
with open("wrong_responses_analysis_plots.r", "w") as file:
    file.write(r_script2)

print("R script saved as 'wrong_responses_analysis_plots.r'")